# Actual-batch frozen-MLP adjoint shadow — diagnostic only
One fresh full NF4/PEFT model and exact public batch29 (not learned step29 state). Unchanged accepted dense starter-half MLP forward; actual native full-M dX remains downstream. Compact live-row dX is a discarded numerical/eligibility witness. Native LoRA dropout and original padded forward/metrics remain.
No optimizer step, full baseline replay, training loop, profiler, score, peak-memory or throughput claim. No certified exceptional-range or production claim. COMPLETED means recorded diagnostic evidence, never numerical/task PASS. Prepared OFFLINE while server is down; execution remains held until explicit user resume.


In [ ]:
import os, base64, hashlib, json, tempfile
from pathlib import Path
assert os.environ.get('SOUP_REPO') and os.environ.get('SOUP_RESULTS'), 'Requires the deployed prepared job environment; no manual path fallback'
assert Path('/kaggle/working').is_dir(), 'Prepared working directory missing'
MODE = 'actual_batch_native_return_mlp_shadow_v1'
PAYLOAD = {'runtime_probe.py': 'IiIiUmVhZC1vbmx5LCBzaG9ydC1saXZlZCBkZXBsb3llZC10ZW1wbGF0ZSBnYXRlLiBOZXZlciBwcmludHMgc2VjcmV0IHZhbHVlcy4iIiIKaW1wb3J0IGltcG9ydGxpYi5tZXRhZGF0YSBhcyBtZXRhZGF0YSxqc29uLG9zLHBsYXRmb3JtLHN1YnByb2Nlc3Msc3lzCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApFWFBFQ1RFRD17J3RvcmNoJzonMi4xMS4wK2N1MTI4JywndHJpdG9uJzonMy42LjAnLCdudW1weSc6JzIuMy41Jywnc291cC1jbGknOicwLjc3LjAnLCd0cmFuc2Zvcm1lcnMnOic1LjE2LjEnLCdwZWZ0JzonMC4yMC4wJywndHJsJzonMC4yOS4xJywnYml0c2FuZGJ5dGVzJzonMC41MC4yJywnYWNjZWxlcmF0ZSc6JzEuMTQuMCcsJ2RhdGFzZXRzJzonNC44LjUnfQoKZGVmIG1haW4oKToKIG91dD1QYXRoKHN5cy5hcmd2WzFdKTtyPXsncGFzc2VkJzpGYWxzZSwnZ3B1X3dvcmtsb2FkX2V4ZWN1dGVkJzpGYWxzZSwnc2NvcmUnOk5vbmUsJ2FkbWlzc2lvbic6J1VOS05PV04nfQogdHJ5OgogIHZlcnNpb25zPXtrOm1ldGFkYXRhLnZlcnNpb24oaykgZm9yIGsgaW4gRVhQRUNURUR9CiAgci51cGRhdGUocHl0aG9uPXBsYXRmb3JtLnB5dGhvbl92ZXJzaW9uKCkscGFja2FnZXM9dmVyc2lvbnMsdWlkPW9zLmdldHVpZCgpLGNwdV9hZmZpbml0eT1zb3J0ZWQob3Muc2NoZWRfZ2V0YWZmaW5pdHkoMCkpKQogIGFzc2VydCBwbGF0Zm9ybS5weXRob25fdmVyc2lvbigpPT0nMy4xMy4xNScsJ1B5dGhvbiBwaW4gbWlzbWF0Y2gnCiAgYXNzZXJ0IHZlcnNpb25zPT1FWFBFQ1RFRCwnRXhhY3QgcGFja2FnZSBwaW5zIG1pc21hdGNoJwogIGFzc2VydCBvcy5nZXR1aWQoKT09MTAwMDEsJ0RlcGxveWVkIHJ1bnRpbWUgdWlkIG1pc21hdGNoJwogIGFzc2VydCBsZW4ob3Muc2NoZWRfZ2V0YWZmaW5pdHkoMCkpPT00LCdGb3VyIGFzc2lnbmVkIENQVXMgcmVxdWlyZWQnCiAgZm9yIHAsd2FudGVkIGluIFsoJ21lbW9yeS5tYXgnLHN0cigzMCoxMDI0KiozKSksKCdtZW1vcnkuc3dhcC5tYXgnLCcwJyksKCdjcHVzZXQuY3B1cy5lZmZlY3RpdmUnLCcwLTMnKV06CiAgIGFjdHVhbD0oUGF0aCgnL3N5cy9mcy9jZ3JvdXAnKS9wKS5yZWFkX3RleHQoKS5zdHJpcCgpO3JbcF09YWN0dWFsCiAgIGFzc2VydCBhY3R1YWw9PXdhbnRlZCwnRGVwbG95ZWQgY2dyb3VwIG1pc21hdGNoOiAnK3AKICBhc3NlcnQgbm90IFBhdGgoJy92YXIvcnVuL2RvY2tlci5zb2NrJykuZXhpc3RzKCksJ1VuZXhwZWN0ZWQgRG9ja2VyIHNvY2tldCcKICBhc3NlcnQgbm90IFBhdGgoJy9ldGMvc291cC1xdWV1ZS93b3JrZXIuZW52JykuZXhpc3RzKCksJ1VuZXhwZWN0ZWQgd29ya2VyIGVudmlyb25tZW50IGZpbGUnCiAgYXNzZXJ0IG5vdCBhbnkoJ1RPS0VOJyBpbiBrIGFuZCBrIT0nVE9LRU5JWkVSU19QQVJBTExFTElTTScgZm9yIGsgaW4gb3MuZW52aXJvbiksJ1Rva2VuLWJlYXJpbmcgZW52aXJvbm1lbnQgcmVqZWN0ZWQgd2l0aG91dCByZWFkaW5nIGl0cyB2YWx1ZScKICBhc3NlcnQgbm90IG9zLmVudmlyb24uZ2V0KCdUUklUT05fSU5URVJQUkVUJyksJ0ludGVycHJldGVyIG1vZGUgZm9yYmlkZGVuJwogIGltcG9ydCB0b3JjaCx0cml0b24sbnVtcHkKICBhc3NlcnQgdG9yY2guX192ZXJzaW9uX189PScyLjExLjArY3UxMjgnIGFuZCB0b3JjaC52ZXJzaW9uLmN1ZGE9PScxMi44JwogIGFzc2VydCB0cml0b24uX192ZXJzaW9uX189PSczLjYuMCcgYW5kIG51bXB5Ll9fdmVyc2lvbl9fPT0nMi4zLjUnCiAgYXNzZXJ0IHRvcmNoLmN1ZGEuZGV2aWNlX2NvdW50KCk9PTEgYW5kIHRvcmNoLmN1ZGEuZ2V0X2RldmljZV9uYW1lKDApPT0nVGVzbGEgVDQnCiAgYXNzZXJ0IHRvcmNoLmN1ZGEuZ2V0X2RldmljZV9jYXBhYmlsaXR5KDApPT0oNyw1KQogIHIudXBkYXRlKGdwdV9uYW1lPXRvcmNoLmN1ZGEuZ2V0X2RldmljZV9uYW1lKDApLGN1ZGFfYnVpbGQ9dG9yY2gudmVyc2lvbi5jdWRhLHRvcmNoX2dpdD10b3JjaC52ZXJzaW9uLmdpdF92ZXJzaW9uKQogIGZyb20gaHVnZ2luZ2ZhY2VfaHViLmNvbnN0YW50cyBpbXBvcnQgSEZfSFVCX0NBQ0hFCiAgclsncmVzb2x2ZWRfcHVibGljX2h1Yl9jYWNoZSddPXN0cihQYXRoKEhGX0hVQl9DQUNIRSkucmVzb2x2ZSgpKQogIHJbJ3Bhc3NlZCddPVRydWUKIGV4Y2VwdCBCYXNlRXhjZXB0aW9uIGFzIGU6clsnZXJyb3InXT1yZXByKGUpCiBvdXQud3JpdGVfdGV4dChqc29uLmR1bXBzKHIsaW5kZW50PTIpKydcbicpO3ByaW50KGpzb24uZHVtcHMocixpbmRlbnQ9MikpCiByYWlzZSBTeXN0ZW1FeGl0KDAgaWYgclsncGFzc2VkJ10gZWxzZSAxKQppZiBfX25hbWVfXz09J19fbWFpbl9fJzptYWluKCkK', 'process_guard.py': 'IiIiSXNvbGF0ZWQgTGludXggc3VicmVhcGVyIGd1YXJkaWFuLiBPd25zIGNoaWxkcmVuIGFuZCBob2xkcyB0aGUgR1BVIGxvY2suCgpUaGUgZ3VhcmRpYW4gd2F0Y2hlcyBhIHBhcmVudC1saXZlbmVzcyBwaXBlOiBkcml2ZXIgU0lHS0lMTCBvciBvdGhlciBhYnJ1cHQKZGVhdGggY2xvc2VzIHRoZSBwaXBlIGFuZCB0cmlnZ2VycyBxdWFyYW50aW5lZCBjbGVhbnVwLiBQSURGRCBzaWduYWxpbmcgaW4gdGhlCnNoYXJlZCB0YXNrIGhlbHBlciBwcmV2ZW50cyBQSUQgcmV1c2UgcmFjZXM7IGRlc2NlbmRhbnRzIGFyZSBvbmx5IHRoaXMgZGVkaWNhdGVkCmd1YXJkaWFuJ3MgY2hpbGRyZW4sIGluY2x1ZGluZyBuZXdseSBhZG9wdGVkIG9ycGhhbnMgZnJvbSBuZXN0ZWQgc2Vzc2lvbnMuClRoaXMgaXMgdGFzayBvcmNoZXN0cmF0aW9uLCBuZXZlciBzb2x1dGlvbi8gd29yayBvciBrZXJuZWwgcGFydGljaXBhdGlvbi4KIiIiCmltcG9ydCBqc29uCmltcG9ydCBvcwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKaW1wb3J0IHNlbGVjdAppbXBvcnQgc2lnbmFsCmltcG9ydCBzdWJwcm9jZXNzCmltcG9ydCBzeXMKaW1wb3J0IHRpbWUKCmltcG9ydCBvd25lZF9wcm9jZXNzX3RyZWUgYXMgdHJlZQoKCmRlZiBxdWFyYW50aW5lKHBhdGgsIHJlYXNvbik6CiAgICAiIiJGaXJzdCBmYWlsdXJlIHdpbnM7IG5ldmVyIG92ZXJ3cml0ZSBvciBhdXRvbWF0aWNhbGx5IGNsZWFyIGEgbWFya2VyLiIiIgogICAgdHJ5OgogICAgICAgIGRlc2NyaXB0b3IgPSBvcy5vcGVuKHBhdGgsIG9zLk9fV1JPTkxZIHwgb3MuT19DUkVBVCB8IG9zLk9fRVhDTCB8IG9zLk9fTk9GT0xMT1csIDBvNjAwKQogICAgZXhjZXB0IEZpbGVFeGlzdHNFcnJvcjoKICAgICAgICByZXR1cm4KICAgIHdpdGggb3MuZmRvcGVuKGRlc2NyaXB0b3IsICJ3IikgYXMgc3RyZWFtOgogICAgICAgIGpzb24uZHVtcCh7InN0YXR1cyI6ICJPUEVSQVRPUl9SRUNPTkNJTElBVElPTl9SRVFVSVJFRCIsICJyZWFzb24iOiByZWFzb24sCiAgICAgICAgICAgICAgICAgICAiZ3VhcmRpYW5fcGlkIjogb3MuZ2V0cGlkKCksICJjcmVhdGVkX3VuaXgiOiB0aW1lLnRpbWUoKX0sIHN0cmVhbSkKICAgICAgICBzdHJlYW0ud3JpdGUoIlxuIikKICAgICAgICBzdHJlYW0uZmx1c2goKQogICAgICAgIG9zLmZzeW5jKHN0cmVhbS5maWxlbm8oKSkKCgpkZWYgX3dyaXRlX3JlY2VpcHQocGF0aCwgdmFsdWUpOgogICAgcGF0aCA9IFBhdGgocGF0aCkKICAgIHBlbmRpbmcgPSBwYXRoLndpdGhfbmFtZShwYXRoLm5hbWUgKyAiLnBlbmRpbmciKQogICAgcGVuZGluZy53cml0ZV90ZXh0KGpzb24uZHVtcHModmFsdWUsIGluZGVudD0yKSArICJcbiIpCiAgICBwZW5kaW5nLnJlcGxhY2UocGF0aCkKCgpkZWYgZ3VhcmRpYW5fbWFpbigpOgogICAgY29uZmlnID0ganNvbi5sb2FkcyhzeXMuc3RkaW4ucmVhZGxpbmUoKSkKICAgIHJlYWRfZmQsIGxvY2tfZmQgPSBjb25maWdbInBhcmVudF9waXBlX2ZkIl0sIGNvbmZpZ1sibG9ja19mZCJdCiAgICBzdGFydGVkLCByZXF1ZXN0ZWRfc2lnbmFsLCBwcm9jZXNzID0gdGltZS5tb25vdG9uaWMoKSwgW05vbmVdLCBOb25lCiAgICByZXN1bHQgPSB7ImV4aXRfY29kZSI6IDEyNSwgInJvb3RfZXhpdF9jb2RlIjogTm9uZSwgInRpbWVkX291dCI6IEZhbHNlLAogICAgICAgICAgICAgICJpbnRlcnJ1cHRlZCI6IEZhbHNlLCAiY2xlYW51cF9jb25maXJtZWQiOiBGYWxzZSwKICAgICAgICAgICAgICAicmVhc29uIjogTm9uZSwgImd1YXJkaWFuX3BpZCI6IG9zLmdldHBpZCgpfQogICAgZGVmIHJlcXVlc3Qoc2lnbnVtLCBmcmFtZSk6CiAgICAgICAgcmVxdWVzdGVkX3NpZ25hbFswXSA9IHNpZ251bQogICAgZm9yIHNpZyBpbiAoc2lnbmFsLlNJR0lOVCwgc2lnbmFsLlNJR1RFUk0pOgogICAgICAgIHNpZ25hbC5zaWduYWwoc2lnLCByZXF1ZXN0KQogICAgdHJ5OgogICAgICAgIHRyZWUucHJlcGFyZSgpICAjIHJlcXVpcmVkIGJlZm9yZSBhbnkgY2hpbGQ7IG5ldmVyIHNpbGVudGx5IGRlZ3JhZGUKICAgICAgICB3aXRoIFBhdGgoY29uZmlnWyJsb2ciXSkub3BlbigieCIpIGFzIHN0cmVhbToKICAgICAgICAgICAgcHJvY2VzcyA9IHN1YnByb2Nlc3MuUG9wZW4oY29uZmlnWyJjb21tYW5kIl0sIGN3ZD1jb25maWdbImN3ZCJdLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBzdGRvdXQ9c3RyZWFtLCBzdGRlcnI9c3VicHJvY2Vzcy5TVERPVVQsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHN0YXJ0X25ld19zZXNzaW9uPVRydWUpCiAgICAgICAgICAgIHdoaWxlIFRydWU6CiAgICAgICAgICAgICAgICByZWFkeSwgXywgXyA9IHNlbGVjdC5zZWxlY3QoW3JlYWRfZmRdLCBbXSwgW10sIDAuMDUpCiAgICAgICAgICAgICAgICBpZiByZWFkeSBhbmQgb3MucmVhZChyZWFkX2ZkLCAxKSA9PSBiIiI6CiAgICAgICAgICAgICAgICAgICAgcmVzdWx0LnVwZGF0ZShyZWFzb249InBhcmVudF9saXZlbmVzc19waXBlX2Nsb3NlZCIsIGludGVycnVwdGVkPVRydWUpCiAgICAgICAgICAgICAgICAgICAgYnJlYWsKICAgICAgICAgICAgICAgIGlmIHJlcXVlc3RlZF9zaWduYWxbMF0gaXMgbm90IE5vbmU6CiAgICAgICAgICAgICAgICAgICAgcmVzdWx0LnVwZGF0ZShyZWFzb249Imd1YXJkaWFuX3NpZ25hbF8iICsgc3RyKHJlcXVlc3RlZF9zaWduYWxbMF0pLCBpbnRlcnJ1cHRlZD1UcnVlKQogICAgICAgICAgICAgICAgICAgIGJyZWFrCiAgICAgICAgICAgICAgICBpZiB0aW1lLm1vbm90b25pYygpIC0gc3RhcnRlZCA+PSBjb25maWdbInRpbWVvdXQiXToKICAgICAgICAgICAgICAgICAgICByZXN1bHQudXBkYXRlKHJlYXNvbj0ib3V0ZXJfdGltZW91dCIsIHRpbWVkX291dD1UcnVlKQogICAgICAgICAgICAgICAgICAgIGJyZWFrCiAgICAgICAgICAgICAgICByZXR1cm5jb2RlID0gcHJvY2Vzcy5wb2xsKCkKICAgICAgICAgICAgICAgIGlmIHJldHVybmNvZGUgaXMgbm90IE5vbmU6CiAgICAgICAgICAgICAgICAgICAgcmVzdWx0WyJyb290X2V4aXRfY29kZSJdID0gcmV0dXJuY29kZQogICAgICAgICAgICAgICAgICAgIGlmIHRyZWUuZGVzY2VuZGFudHMoKToKICAgICAgICAgICAgICAgICAgICAgICAgcmVzdWx0WyJyZWFzb24iXSA9ICJyb290X2V4aXRlZF93aXRoX293bmVkX2Rlc2NlbmRhbnRzIgogICAgICAgICAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICAgICAgICAgIHJlc3VsdC51cGRhdGUoZXhpdF9jb2RlPXJldHVybmNvZGUsIGNsZWFudXBfY29uZmlybWVkPVRydWUpCiAgICAgICAgICAgICAgICAgICAgYnJlYWsKICAgICAgICAgICAgaWYgcmVzdWx0WyJyZWFzb24iXSBpcyBub3QgTm9uZToKICAgICAgICAgICAgICAgIHF1YXJhbnRpbmUoY29uZmlnWyJtYXJrZXIiXSwgcmVzdWx0WyJyZWFzb24iXSkKICAgICAgICAgICAgICAgICMgY2xlYW51cCBpZ25vcmVzIHJlcGVhdGVkIElOVC9URVJNLCBzdG9wcy9yZXNjYW5zL2tpbGxzIGJ5CiAgICAgICAgICAgICAgICAjIHBpZGZkIGFuZCB3YWl0cy9yZWFwcyB3aXRob3V0IGFueSB0aW1lb3V0LXVubG9jayBlc2NhcGUuCiAgICAgICAgICAgICAgICB0cmVlLmNsZWFudXAocHJvY2VzcykKICAgICAgICAgICAgICAgIHJlc3VsdC51cGRhdGUocm9vdF9leGl0X2NvZGU9cHJvY2Vzcy5yZXR1cm5jb2RlLCBjbGVhbnVwX2NvbmZpcm1lZD1UcnVlKQogICAgICAgICAgICAgICAgaWYgcmVzdWx0WyJ0aW1lZF9vdXQiXToKICAgICAgICAgICAgICAgICAgICByZXN1bHRbImV4aXRfY29kZSJdID0gMTI0CiAgICAgICAgICAgICAgICBlbGlmIHJlc3VsdFsiaW50ZXJydXB0ZWQiXToKICAgICAgICAgICAgICAgICAgICByZXN1bHRbImV4aXRfY29kZSJdID0gMTMwCiAgICBleGNlcHQgQmFzZUV4Y2VwdGlvbiBhcyBleGM6CiAgICAgICAgcmVzdWx0LnVwZGF0ZShyZWFzb249Imd1YXJkaWFuX2V4Y2VwdGlvbjoiICsgcmVwcihleGMpLCBleGl0X2NvZGU9MTI1KQogICAgICAgIHRyeToKICAgICAgICAgICAgcXVhcmFudGluZShjb25maWdbIm1hcmtlciJdLCByZXN1bHRbInJlYXNvbiJdKQogICAgICAgIGZpbmFsbHk6CiAgICAgICAgICAgIGlmIHByb2Nlc3MgaXMgbm90IE5vbmU6CiAgICAgICAgICAgICAgICB0cmVlLmNsZWFudXAocHJvY2VzcykKICAgICAgICAgICAgcmVzdWx0WyJjbGVhbnVwX2NvbmZpcm1lZCJdID0gbm90IHRyZWUuZGVzY2VuZGFudHMoKQogICAgZmluYWxseToKICAgICAgICByZXN1bHRbIm91dGVyX3dhbGxfcyJdID0gdGltZS5tb25vdG9uaWMoKSAtIHN0YXJ0ZWQKICAgICAgICBfd3JpdGVfcmVjZWlwdChjb25maWdbInJlY2VpcHQiXSwgcmVzdWx0KQogICAgICAgIG9zLmNsb3NlKHJlYWRfZmQpCiAgICAgICAgIyBUaGUgaW5oZXJpdGVkIG9wZW4tZmlsZSBkZXNjcmlwdGlvbiByZXRhaW5zIHRoZSBjYW5vbmljYWwgZmxvY2sgZXZlbgogICAgICAgICMgaWYgdGhlIGRyaXZlciBoYXMgZGllZC4gQ2xvc2UgT05MWSBhZnRlciBvd25lZC10cmVlIGNsZWFudXAgY29tcGxldGVzLgogICAgICAgIG9zLmNsb3NlKGxvY2tfZmQpCiAgICByZXR1cm4gcmVzdWx0WyJleGl0X2NvZGUiXQoKCmRlZiBydW4oY29tbWFuZCwgKiwgY3dkLCBlbnYsIGxvZywgdGltZW91dCwgbG9ja19mZCwKICAgICAgICBtYXJrZXI9Ii90bXAvc291cC1ncHUwLmxvY2suYmxvY2tlZCIpOgogICAgIiIiRGVkaWNhdGVkIGRyaXZlciBBUEkuIEFueSBCYXNlRXhjZXB0aW9uIGFza3MgZ3VhcmRpYW4gY2xlYW51cCBhbmQgd2FpdHMuIiIiCiAgICByZWNlaXB0ID0gUGF0aChzdHIobG9nKSArICIucHJvY2Vzcy5qc29uIikucmVzb2x2ZSgpCiAgICBpZiByZWNlaXB0LmV4aXN0cygpOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiUHJvY2VzcyByZWNlaXB0IGFscmVhZHkgZXhpc3RzOyByZWZ1c2UgdG8gb3ZlcndyaXRlIikKICAgIHJlYWRfZmQsIHdyaXRlX2ZkID0gb3MucGlwZSgpCiAgICBjb25maWcgPSB7ImNvbW1hbmQiOiBbc3RyKHgpIGZvciB4IGluIGNvbW1hbmRdLCAiY3dkIjogc3RyKFBhdGgoY3dkKS5yZXNvbHZlKCkpLAogICAgICAgICAgICAgICJsb2ciOiBzdHIoUGF0aChsb2cpLnJlc29sdmUoKSksICJ0aW1lb3V0IjogdGltZW91dCwKICAgICAgICAgICAgICAicmVjZWlwdCI6IHN0cihyZWNlaXB0KSwgInBhcmVudF9waXBlX2ZkIjogcmVhZF9mZCwKICAgICAgICAgICAgICAibG9ja19mZCI6IGxvY2tfZmQsICJtYXJrZXIiOiBzdHIobWFya2VyKX0KICAgIGd1YXJkaWFuID0gTm9uZQogICAgaW50ZXJydXB0ZWQgPSBOb25lCiAgICBkZWYgcmVxdWVzdChzaWdudW0sIGZyYW1lKToKICAgICAgICByYWlzZSBJbnRlcnJ1cHRlZEVycm9yKCJEcml2ZXIgcmVjZWl2ZWQgc2lnbmFsICIgKyBzdHIoc2lnbnVtKSkKICAgIG9sZF9oYW5kbGVycyA9IHtzaWc6IHNpZ25hbC5zaWduYWwoc2lnLCByZXF1ZXN0KSBmb3Igc2lnIGluIChzaWduYWwuU0lHSU5ULCBzaWduYWwuU0lHVEVSTSl9CiAgICB0cnk6CiAgICAgICAgZ3VhcmRpYW4gPSBzdWJwcm9jZXNzLlBvcGVuKFtzeXMuZXhlY3V0YWJsZSwgc3RyKFBhdGgoX19maWxlX18pLnJlc29sdmUoKSksICItLWd1YXJkaWFuIl0sCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHN0ZGluPXN1YnByb2Nlc3MuUElQRSwgZW52PWVudiwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgcGFzc19mZHM9KHJlYWRfZmQsIGxvY2tfZmQpLCBzdGFydF9uZXdfc2Vzc2lvbj1UcnVlLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICB0ZXh0PVRydWUpCiAgICAgICAgb3MuY2xvc2UocmVhZF9mZCkKICAgICAgICByZWFkX2ZkID0gTm9uZQogICAgICAgIGd1YXJkaWFuLnN0ZGluLndyaXRlKGpzb24uZHVtcHMoY29uZmlnKSArICJcbiIpCiAgICAgICAgZ3VhcmRpYW4uc3RkaW4uY2xvc2UoKQogICAgICAgIGd1YXJkaWFuLndhaXQoKQogICAgZXhjZXB0IEJhc2VFeGNlcHRpb24gYXMgZXhjOgogICAgICAgIGludGVycnVwdGVkID0gcmVwcihleGMpCiAgICAgICAgIyBJZ25vcmUgcmVwZWF0ZWQgc3RvcCBzaWduYWxzIHVudGlsIHRoZSBndWFyZGlhbiBjb25maXJtcyBjbGVhbnVwLgogICAgICAgIGZvciBzaWcgaW4gb2xkX2hhbmRsZXJzOgogICAgICAgICAgICBzaWduYWwuc2lnbmFsKHNpZywgc2lnbmFsLlNJR19JR04pCiAgICAgICAgaWYgd3JpdGVfZmQgaXMgbm90IE5vbmU6CiAgICAgICAgICAgIG9zLmNsb3NlKHdyaXRlX2ZkKQogICAgICAgICAgICB3cml0ZV9mZCA9IE5vbmUKICAgICAgICBpZiBndWFyZGlhbiBpcyBub3QgTm9uZToKICAgICAgICAgICAgaWYgZ3VhcmRpYW4uc3RkaW4gaXMgbm90IE5vbmUgYW5kIG5vdCBndWFyZGlhbi5zdGRpbi5jbG9zZWQ6CiAgICAgICAgICAgICAgICBndWFyZGlhbi5zdGRpbi5jbG9zZSgpCiAgICAgICAgICAgIGd1YXJkaWFuLndhaXQoKQogICAgZmluYWxseToKICAgICAgICBpZiByZWFkX2ZkIGlzIG5vdCBOb25lOgogICAgICAgICAgICBvcy5jbG9zZShyZWFkX2ZkKQogICAgICAgIGlmIHdyaXRlX2ZkIGlzIG5vdCBOb25lOgogICAgICAgICAgICBvcy5jbG9zZSh3cml0ZV9mZCkKICAgICAgICBmb3Igc2lnLCBoYW5kbGVyIGluIG9sZF9oYW5kbGVycy5pdGVtcygpOgogICAgICAgICAgICBzaWduYWwuc2lnbmFsKHNpZywgaGFuZGxlcikKICAgIGlmIG5vdCByZWNlaXB0LmlzX2ZpbGUoKToKICAgICAgICBxdWFyYW50aW5lKG1hcmtlciwgImd1YXJkaWFuX21pc3NpbmdfcmVjZWlwdCIpCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJHdWFyZGlhbiByZWNlaXB0IG1pc3Npbmc7IEdQVSBxdWFyYW50aW5lIHJlcXVpcmVzIG9wZXJhdG9yIHJlY29uY2lsaWF0aW9uIikKICAgIHJlc3VsdCA9IGpzb24ubG9hZHMocmVjZWlwdC5yZWFkX3RleHQoKSkKICAgIGlmIGludGVycnVwdGVkIGlzIG5vdCBOb25lOgogICAgICAgIHJlc3VsdC51cGRhdGUocGFyZW50X2V4Y2VwdGlvbj1pbnRlcnJ1cHRlZCwgaW50ZXJydXB0ZWQ9VHJ1ZSkKICAgICAgICBpZiByZXN1bHRbImV4aXRfY29kZSJdID09IDA6CiAgICAgICAgICAgIHJlc3VsdFsiZXhpdF9jb2RlIl0gPSAxMzAKICAgICAgICBxdWFyYW50aW5lKG1hcmtlciwgImRyaXZlcl9pbnRlcnJ1cHRpb246IiArIGludGVycnVwdGVkKQogICAgaWYgbm90IHJlc3VsdFsiY2xlYW51cF9jb25maXJtZWQiXToKICAgICAgICBxdWFyYW50aW5lKG1hcmtlciwgImd1YXJkaWFuX3VuY2VydGFpbl9jbGVhbnVwIikKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIkd1YXJkaWFuIGNsZWFudXAgdW5jZXJ0YWluOyBxdWFyYW50aW5lIHJldGFpbmVkIikKICAgIHJldHVybiByZXN1bHQKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgaWYgc3lzLmFyZ3ZbMTpdICE9IFsiLS1ndWFyZGlhbiJdOgogICAgICAgIHJhaXNlIFN5c3RlbUV4aXQoIkludGVybmFsIGd1YXJkaWFuIGVudHJ5cG9pbnQgb25seSIpCiAgICByYWlzZSBTeXN0ZW1FeGl0KGd1YXJkaWFuX21haW4oKSkK', 'owned_process_tree.py': 'IyBTaGFyZWQgdGFzayBvcmNoZXN0cmF0aW9uIGZyb20gdGhlIExvUkEgbGFuZTsgbm8ga2VybmVsIG93bmVyc2hpcCBjcmVkaXQuCiIiIkxpbnV4LW9ubHkgZGVkaWNhdGVkLXJ1bm5lciBjbGVhbnVwOyBubyBHUFUgaW1wb3J0cyBvciBiYWNrZ3JvdW5kIGFjdGl2aXR5LgoKT3JpZ2luYWwgdGFzay1vcmNoZXN0cmF0aW9uIGhlbHBlci4gTm90IHBhcnQgb2Ygc29sdXRpb24vIG9yIG93bi1rZXJuZWwgY3JlZGl0LgoiIiIKaW1wb3J0IGN0eXBlcwppbXBvcnQgb3MKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmltcG9ydCBzaWduYWwKaW1wb3J0IHN1YnByb2Nlc3MKaW1wb3J0IHN5cwppbXBvcnQgdGltZQoKCmRlZiBwcmVwYXJlKCk6CiAgICAiIiJGYWlsIGJlZm9yZSBHUFUgd29yayBpZiByYWNlLXNhZmUgcHJvY2VzcyBvd25lcnNoaXAgaXMgdW5hdmFpbGFibGUuIiIiCiAgICBpZiBub3QgaGFzYXR0cihvcywncGlkZmRfb3BlbicpIG9yIG5vdCBoYXNhdHRyKHNpZ25hbCwncGlkZmRfc2VuZF9zaWduYWwnKToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoJ0xpbnV4IHBpZGZkIHByb2Nlc3Mgb3duZXJzaGlwIHJlcXVpcmVkJykKICAgIGZkPW9zLnBpZGZkX29wZW4ob3MuZ2V0cGlkKCkpO29zLmNsb3NlKGZkKQogICAgbGliYz1jdHlwZXMuQ0RMTChOb25lLHVzZV9lcnJubz1UcnVlKQogICAgaWYgbGliYy5wcmN0bCgzNiwxLDAsMCwwKSE9MDoKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoJ0Nhbm5vdCBlbmFibGUgY2hpbGQgc3VicmVhcGVyOyBubyBHUFUgam9iIHN0YXJ0ZWQnKQogICAgZW5hYmxlZD1jdHlwZXMuY19pbnQoMCkKICAgIGlmIGxpYmMucHJjdGwoMzcsY3R5cGVzLmJ5cmVmKGVuYWJsZWQpLDAsMCwwKSE9MCBvciBlbmFibGVkLnZhbHVlIT0xOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcignQ2hpbGQgc3VicmVhcGVyIHZlcmlmaWNhdGlvbiBmYWlsZWQnKQoKCmRlZiBpZGVudGl0eShwaWQpOgogICAgdHJ5OgogICAgICAgIGZpZWxkcz1QYXRoKGYnL3Byb2Mve3BpZH0vc3RhdCcpLnJlYWRfdGV4dCgpLnJzcGxpdCgnKScsMSlbMV0uc3BsaXQoKQogICAgICAgIHJldHVybiBpbnQoZmllbGRzWzFdKSxpbnQoZmllbGRzWzE5XSksZmllbGRzWzBdCiAgICBleGNlcHQgKE9TRXJyb3IsVmFsdWVFcnJvcixJbmRleEVycm9yKTpyZXR1cm4gTm9uZQoKCmRlZiBkZXNjZW5kYW50cygpOgogICAgcmVjb3Jkcz17fQogICAgZm9yIHAgaW4gUGF0aCgnL3Byb2MnKS5pdGVyZGlyKCk6CiAgICAgICAgaWYgcC5uYW1lLmlzZGlnaXQoKToKICAgICAgICAgICAgcmVjPWlkZW50aXR5KGludChwLm5hbWUpKQogICAgICAgICAgICBpZiByZWMgaXMgbm90IE5vbmU6cmVjb3Jkc1tpbnQocC5uYW1lKV09cmVjCiAgICBvd25lZD17b3MuZ2V0cGlkKCl9CiAgICB3aGlsZSBUcnVlOgogICAgICAgIGFkZD17cGlkIGZvciBwaWQsKHBhcmVudCxfLF8pIGluIHJlY29yZHMuaXRlbXMoKSBpZiBwYXJlbnQgaW4gb3duZWR9LW93bmVkCiAgICAgICAgaWYgbm90IGFkZDpicmVhawogICAgICAgIG93bmVkLnVwZGF0ZShhZGQpCiAgICByZXR1cm4ge3BpZDpyZWNvcmRzW3BpZF0gZm9yIHBpZCBpbiBvd25lZCBpZiBwaWQhPW9zLmdldHBpZCgpIGFuZCBwaWQgaW4gcmVjb3Jkc30KCgpkZWYgc2lnbmFsX2V4YWN0KHBpZCxiaXJ0aCxzaWcpOgogICAgIiIiTmV2ZXIgc2lnbmFsIGEgcmVjeWNsZWQgUElELiBUaGUgRkQgcGlucyBpZGVudGl0eSB0aHJvdWdoIHRoZSBzZW5kLiIiIgogICAgdHJ5OmZkPW9zLnBpZGZkX29wZW4ocGlkKQogICAgZXhjZXB0IFByb2Nlc3NMb29rdXBFcnJvcjpyZXR1cm4KICAgIHRyeToKICAgICAgICBub3c9aWRlbnRpdHkocGlkKQogICAgICAgIGlmIG5vdyBpcyBub3QgTm9uZSBhbmQgbm93WzFdPT1iaXJ0aDoKICAgICAgICAgICAgdHJ5OnNpZ25hbC5waWRmZF9zZW5kX3NpZ25hbChmZCxzaWcsTm9uZSwwKQogICAgICAgICAgICBleGNlcHQgUHJvY2Vzc0xvb2t1cEVycm9yOnBhc3MKICAgIGZpbmFsbHk6b3MuY2xvc2UoZmQpCgoKZGVmIF90cnlfc2lnbmFsKHBpZCxiaXJ0aCxzaWcpOgogICAgdHJ5OnNpZ25hbF9leGFjdChwaWQsYmlydGgsc2lnKQogICAgZXhjZXB0IE9TRXJyb3IgYXMgZXhjOgogICAgICAgICMgRG8gbm90IHVud2luZC9yZWxlYXNlIEdQVSBvd25lcnNoaXAgd2hlbiBjbGVhbnVwIGlzIHVuY2VydGFpbi4KICAgICAgICBxdWFyYW50aW5lKCdDbGVhbnVwIHNpZ25hbCBmYWlsZWQ6ICcrcmVwcihleGMpKQogICAgICAgIHByaW50KCdTT1VQX0NMRUFOVVBfUEVORElORyAnK3JlcHIoZXhjKSxmaWxlPXN5cy5zdGRlcnIsZmx1c2g9VHJ1ZSkKCgpkZWYgY2xlYW51cChwcm9jKToKICAgIHNhdmVkPXtzOnNpZ25hbC5zaWduYWwocyxzaWduYWwuU0lHX0lHTikgZm9yIHMgaW4gKHNpZ25hbC5TSUdJTlQsc2lnbmFsLlNJR1RFUk0pfQogICAgdHJ5OgogICAgICAgIGZyb3plbj1zZXQoKQogICAgICAgIHdoaWxlIFRydWU6CiAgICAgICAgICAgIG9ic2VydmVkPWRlc2NlbmRhbnRzKCkKICAgICAgICAgICAgbmV3PXsocGlkLHZbMV0pIGZvciBwaWQsdiBpbiBvYnNlcnZlZC5pdGVtcygpfS1mcm96ZW4KICAgICAgICAgICAgaWYgbm90IG5ldzpicmVhawogICAgICAgICAgICBmb3IgcGlkLGJpcnRoIGluIG5ldzpfdHJ5X3NpZ25hbChwaWQsYmlydGgsc2lnbmFsLlNJR1NUT1ApCiAgICAgICAgICAgIGZyb3plbi51cGRhdGUobmV3KQogICAgICAgIGZvciBwaWQsYmlydGggaW4gZnJvemVuOl90cnlfc2lnbmFsKHBpZCxiaXJ0aCxzaWduYWwuU0lHS0lMTCkKICAgICAgICBwcm9jLndhaXQoKQogICAgICAgICMgTm8gdGltZW91dC1iYXNlZCByZWxlYXNlOiBpZiBhIEdQVSBjaGlsZCBpcyBzdHVjayBpbiBrZXJuZWwgZXhpdCwKICAgICAgICAjIHJldGFpbiB0aGlzIHN1cGVydmlzb3IgYW5kIGl0cyBsb2NrIHVudGlsIGV4aXQgaXMgYWN0dWFsbHkgb2JzZXJ2ZWQuCiAgICAgICAgd2hpbGUgVHJ1ZToKICAgICAgICAgICAgd2hpbGUgVHJ1ZToKICAgICAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgICAgICBwaWQsXz1vcy53YWl0cGlkKC0xLG9zLldOT0hBTkcpCiAgICAgICAgICAgICAgICAgICAgaWYgbm90IHBpZDpicmVhawogICAgICAgICAgICAgICAgZXhjZXB0IENoaWxkUHJvY2Vzc0Vycm9yOmJyZWFrCiAgICAgICAgICAgIHJlbWFpbmluZz1kZXNjZW5kYW50cygpCiAgICAgICAgICAgIGlmIG5vdCByZW1haW5pbmc6YnJlYWsKICAgICAgICAgICAgZm9yIHBpZCwoXyxiaXJ0aCxfKSBpbiByZW1haW5pbmcuaXRlbXMoKTpfdHJ5X3NpZ25hbChwaWQsYmlydGgsc2lnbmFsLlNJR0tJTEwpCiAgICAgICAgICAgIHRpbWUuc2xlZXAoLjA1KQogICAgZmluYWxseToKICAgICAgICBmb3Igc2lnLGhhbmRsZXIgaW4gc2F2ZWQuaXRlbXMoKTpzaWduYWwuc2lnbmFsKHNpZyxoYW5kbGVyKQoKCmRlZiBydW4oYXJndiwqLGN3ZCxlbnYsc3Rkb3V0LHRpbWVvdXQ9Tm9uZSxzdGRlcnI9c3VicHJvY2Vzcy5TVERPVVQpOgogICAgIiIiUnVuIGluIGRlZGljYXRlZCBzdXBlcnZpc29yIG9ubHk7IGNsZWFudXAgbmVzdGVkIHNlc3Npb25zIG9uIGFueSBmYWlsdXJlLiIiIgogICAgcHJvYz1zdWJwcm9jZXNzLlBvcGVuKGFyZ3YsY3dkPWN3ZCxlbnY9ZW52LHN0ZG91dD1zdGRvdXQsc3RkZXJyPXN0ZGVycixzdGFydF9uZXdfc2Vzc2lvbj1UcnVlKQogICAgdHJ5OgogICAgICAgIGNvZGU9cHJvYy53YWl0KHRpbWVvdXQ9dGltZW91dCkKICAgICAgICBpZiBkZXNjZW5kYW50cygpOgogICAgICAgICAgICBxdWFyYW50aW5lKCdDaGlsZCBleGl0ZWQgbGVhdmluZyBkZXNjZW5kYW50cycpCiAgICAgICAgICAgIGNsZWFudXAocHJvYykKICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCdDaGlsZCBleGl0ZWQgbGVhdmluZyBkZXNjZW5kYW50czsgY2xlYW5lZCBhbmQgcmVqZWN0ZWQnKQogICAgICAgIHJldHVybiBjb2RlCiAgICBleGNlcHQgQmFzZUV4Y2VwdGlvbiBhcyBleGM6CiAgICAgICAgcXVhcmFudGluZSgnT3V0ZXIgdGltZW91dC9pbnRlcnJ1cHRpb24vZXJyb3I6ICcrcmVwcihleGMpKQogICAgICAgIGNsZWFudXAocHJvYykKICAgICAgICByYWlzZQoKCmRlZiBpbnN0YWxsX3NpZ25hbF9oYW5kbGVycygpOgogICAgZGVmIGludGVycnVwdGVkKHNpZ251bSxmcmFtZSk6CiAgICAgICAgcXVhcmFudGluZSgnUnVubmVyIGludGVycnVwdGVkIGJ5IHNpZ25hbCAnK3N0cihzaWdudW0pKQogICAgICAgIHJhaXNlIEludGVycnVwdGVkRXJyb3IoJ1J1bm5lciBpbnRlcnJ1cHRlZCBieSBzaWduYWwgJytzdHIoc2lnbnVtKSkKICAgIGZvciBzaWcgaW4gKHNpZ25hbC5TSUdJTlQsc2lnbmFsLlNJR1RFUk0pOnNpZ25hbC5zaWduYWwoc2lnLGludGVycnVwdGVkKQoKIyBBIGNyYXNoLXN1cnZpdmluZyBtYXJrZXIgaXMgcGxhY2VkIGJlZm9yZSBhbnkgR1BVIHN1YnByb2Nlc3MuIE9ubHkgaXRzIGV4YWN0CiMgb3duZXIgbWF5IGNsZWFyIGl0IG9uIGEgY2xlYW4sIG5vbi1pbnRlcnJ1cHRlZCBleGl0IHdpdGggbm8gbGl2ZSBkZXNjZW5kYW50cy4KIyBJbnRlcnJ1cHRlZC90aW1lZC1vdXQgd29yayBsZWF2ZXMgaXQgZm9yIGV4cGxpY2l0IG9wZXJhdG9yIHJlY29uY2lsaWF0aW9uLgpfTUFSS0VSID0gTm9uZQpfVE9LRU4gPSBOb25lCl9LRUVQX01BUktFUiA9IEZhbHNlCgoKZGVmIGNoZWNrX3F1YXJhbnRpbmUocGF0aCk6CiAgICBpZiBQYXRoKHBhdGgpLmV4aXN0cygpOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcignR1BVMCBxdWFyYW50aW5lZDogcmVjb25jaWxlICcrc3RyKHBhdGgpKycgYmVmb3JlIGFub3RoZXIgcnVuJykKCgpkZWYgYmVnaW5fcXVhcmFudGluZShwYXRoKToKICAgIGdsb2JhbCBfTUFSS0VSLF9UT0tFTixfS0VFUF9NQVJLRVIKICAgIGltcG9ydCBqc29uLHV1aWQKICAgIF9NQVJLRVI9UGF0aChwYXRoKTtfVE9LRU49dXVpZC51dWlkNCgpLmhleDtfS0VFUF9NQVJLRVI9RmFsc2UKICAgIGZkPW9zLm9wZW4oc3RyKF9NQVJLRVIpLG9zLk9fV1JPTkxZfG9zLk9fQ1JFQVR8b3MuT19FWENMfG9zLk9fTk9GT0xMT1csMG82MDApCiAgICB3aXRoIG9zLmZkb3BlbihmZCwndycpIGFzIG91dDoKICAgICAgICBqc29uLmR1bXAoeydvd25lcl90b2tlbic6X1RPS0VOLCdvd25lcl9waWQnOm9zLmdldHBpZCgpLCdzdGF0dXMnOidPV05FRF9HUFVfV09SS19JTl9QUk9HUkVTUyd9LG91dCkKICAgICAgICBvdXQuZmx1c2goKTtvcy5mc3luYyhvdXQuZmlsZW5vKCkpCgoKZGVmIHF1YXJhbnRpbmUocmVhc29uKToKICAgIGdsb2JhbCBfS0VFUF9NQVJLRVIKICAgIF9LRUVQX01BUktFUj1UcnVlCiAgICBpZiBfTUFSS0VSIGlzIE5vbmU6cmV0dXJuCiAgICBpbXBvcnQganNvbgogICAgdHJ5OnZhbHVlPWpzb24ubG9hZHMoX01BUktFUi5yZWFkX3RleHQoKSkKICAgIGV4Y2VwdCAoT1NFcnJvcixWYWx1ZUVycm9yKTpyZXR1cm4KICAgIGlmIHZhbHVlLmdldCgnb3duZXJfdG9rZW4nKSE9X1RPS0VOOnJldHVybgogICAgdmFsdWUudXBkYXRlKHN0YXR1cz0nQkxPQ0tFRF9PUEVSQVRPUl9SRUNPTkNJTElBVElPTl9SRVFVSVJFRCcscmVhc29uPXN0cihyZWFzb24pKQogICAgX01BUktFUi53cml0ZV90ZXh0KGpzb24uZHVtcHModmFsdWUpKydcbicpCgoKZGVmIGZpbmlzaF9xdWFyYW50aW5lKCk6CiAgICBpZiBfTUFSS0VSIGlzIE5vbmUgb3IgX0tFRVBfTUFSS0VSOnJldHVybgogICAgaWYgZGVzY2VuZGFudHMoKToKICAgICAgICBxdWFyYW50aW5lKCdMaXZlIG93bmVkIGRlc2NlbmRhbnRzIHJlbWFpbiBhdCBzdXBlcnZpc29yIGV4aXQnKQogICAgICAgIHJldHVybgogICAgaW1wb3J0IGpzb24KICAgIHRyeTp2YWx1ZT1qc29uLmxvYWRzKF9NQVJLRVIucmVhZF90ZXh0KCkpCiAgICBleGNlcHQgKE9TRXJyb3IsVmFsdWVFcnJvcik6cmV0dXJuCiAgICBpZiB2YWx1ZS5nZXQoJ293bmVyX3Rva2VuJyk9PV9UT0tFTjpfTUFSS0VSLnVubGluaygpCg==', 'prepare_asset_manifest.py': 'IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiJSZWFkL2hhc2ggb25seSB0aGUgc2VsZWN0ZWQgcHVibGljIG1vZGVsIGNhY2hlICsgcHVibGljIHRhc2sgZmlsZXM7IG5vIGRvd25sb2FkLiIiIgppbXBvcnQgYXJncGFyc2UsaGFzaGxpYixqc29uLHJlCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApNT0RFTD0nbW9kZWxzLS1Rd2VuLS1Rd2VuMi41LTEuNUItSW5zdHJ1Y3QnClJFVklTSU9OPSc5ODlhYTc5ODBlNGNmODA2ZjgwYzdmZWYyYjFhZGI3YmM3MWFhMzA2JwpEQVRBPXsnZGF0YS90cmFpbi5qc29ubCc6JzkzZDAzNmQ5MTJiZDgxYzMzN2RjMjk4OTNlMTNhNzZmMzYxNjFkOTZmODk2ZWY0ZTAyNDkyZTJmMWEyMWEwZGQnLCdkYXRhL2V2YWxfcHVibGljLmpzb25sJzonYjdmODViNzhiYTMxODZkY2U0ZGVhYjdjZmI4YmY4ZGNmNzk5OGUxNTE3YzU3NDlhYzBmNTAwNTg4NmM2ZjcwZSd9CmRlZiBzaGEocGF0aCk6CiBoPWhhc2hsaWIuc2hhMjU2KCkKIHdpdGggcGF0aC5vcGVuKCdyYicpIGFzIGY6CiAgZm9yIGIgaW4gaXRlcihsYW1iZGE6Zi5yZWFkKDQqMTAyNCoxMDI0KSxiJycpOmgudXBkYXRlKGIpCiByZXR1cm4gaC5oZXhkaWdlc3QoKQpkZWYgYnVpbGQoY2FjaGUscmVwbyxyZXZpc2lvbj1SRVZJU0lPTik6CiBjYWNoZT1QYXRoKGNhY2hlKS5yZXNvbHZlKCk7cmVwbz1QYXRoKHJlcG8pLnJlc29sdmUoKQogaWYgbm90IHJlLmZ1bGxtYXRjaCgnWzAtOWEtZl17NDB9JyxyZXZpc2lvbik6cmFpc2UgVmFsdWVFcnJvcignUmV2aXNpb24gbXVzdCBiZTQwaGV4JykKIHJvb3Q9Y2FjaGUvTU9ERUwKIGlmIHJvb3QuaXNfc3ltbGluaygpIG9yIG5vdCByb290LmlzX2RpcigpOnJhaXNlIFZhbHVlRXJyb3IoJ1NlbGVjdGVkIG1vZGVsIGNhY2hlIG1pc3Npbmcgb3Igc3ltbGlua2VkJykKIGZvciBkaXJlY3RvcnkgaW4gKHJvb3QvJ3JlZnMnLHJvb3QvJ3NuYXBzaG90cycscm9vdC8nYmxvYnMnKToKICBpZiBkaXJlY3RvcnkuaXNfc3ltbGluaygpIG9yIG5vdCBkaXJlY3RvcnkuaXNfZGlyKCk6cmFpc2UgVmFsdWVFcnJvcignTW9kZWwgbWV0YWRhdGEvYmxvYiBkaXJlY3RvcmllcyBtdXN0IGJlIHJlYWwgbG9jYWwgZGlyZWN0b3JpZXMnKQogcmVmPXJvb3QvJ3JlZnMvbWFpbicKIGlmIHJlZi5pc19zeW1saW5rKCkgb3Igbm90IHJlZi5pc19maWxlKCkgb3IgcmVmLnJlYWRfdGV4dCgpLnN0cmlwKCkhPXJldmlzaW9uOnJhaXNlIFZhbHVlRXJyb3IoJ3JlZnMvbWFpbiBtdXN0IGlkZW50aWZ5IGV4YWN0IHBpbm5lZCByZXZpc2lvbjsgbm8gc2lsZW50IHJldmlzaW9uIHN1YnN0aXR1dGlvbicpCiBzbmFwc2hvdD1yb290LydzbmFwc2hvdHMnL3JldmlzaW9uCiBpZiBzbmFwc2hvdC5pc19zeW1saW5rKCkgb3Igbm90IHNuYXBzaG90LmlzX2RpcigpOnJhaXNlIFZhbHVlRXJyb3IoJ1NuYXBzaG90IGRpcmVjdG9yeSBhYnNlbnQ7IHJlc3RvcmUgdmFsaWRhdGVkIHJldGFpbmVkIGxpbmtzIHNlcGFyYXRlbHksIGRvIG5vdCBndWVzcycpCiBmaWxlcz17c3RyKHJlZi5yZWxhdGl2ZV90byhjYWNoZSkpOnNoYShyZWYpfTttZW1vPXt9CiBmb3IgcCBpbiBzb3J0ZWQoc25hcHNob3Qucmdsb2IoJyonKSk6CiAgaWYgcC5pc19kaXIoKToKICAgaWYgcC5pc19zeW1saW5rKCk6cmFpc2UgVmFsdWVFcnJvcignTGlua2VkIHNuYXBzaG90IGRpcmVjdG9yeSByZWplY3RlZCcpCiAgIGNvbnRpbnVlCiAgcmVzb2x2ZWQ9cC5yZXNvbHZlKHN0cmljdD1UcnVlKQogIGlmIG5vdCByZXNvbHZlZC5pc19yZWxhdGl2ZV90bygocm9vdC8nYmxvYnMnKS5yZXNvbHZlKCkpIG9yIG5vdCByZXNvbHZlZC5pc19maWxlKCk6cmFpc2UgVmFsdWVFcnJvcignU25hcHNob3QgdGFyZ2V0IG11c3QgcmVtYWluIGluc2lkZSBzZWxlY3RlZCBtb2RlbCBibG9icycpCiAgaWYgcmVzb2x2ZWQgbm90IGluIG1lbW86bWVtb1tyZXNvbHZlZF09c2hhKHJlc29sdmVkKQogIGZpbGVzW3N0cihwLnJlbGF0aXZlX3RvKGNhY2hlKSldPW1lbW9bcmVzb2x2ZWRdCiBpZiBsZW4oZmlsZXMpPDI6cmFpc2UgVmFsdWVFcnJvcignRW1wdHkgbW9kZWwgc25hcHNob3QnKQogZGF0YT17fQogZm9yIG5hbWUsd2FudGVkIGluIERBVEEuaXRlbXMoKToKICBwPXJlcG8vbmFtZQogIGlmIG5vdCBwLmlzX2ZpbGUoKSBvciBwLmlzX3N5bWxpbmsoKSBvciBub3QgcC5yZXNvbHZlKCkuaXNfcmVsYXRpdmVfdG8ocmVwbykgb3Igc2hhKHApIT13YW50ZWQ6cmFpc2UgVmFsdWVFcnJvcignUHVibGljIGRhdGEgaGFzaCBtaXNtYXRjaDogJytuYW1lKQogIGRhdGFbbmFtZV09d2FudGVkCiByZXR1cm4geydtb2RlbF9pZCc6J1F3ZW4vUXdlbjIuNS0xLjVCLUluc3RydWN0JywncmV2aXNpb24nOnJldmlzaW9uLCdmaWxlcyc6ZmlsZXN9LGRhdGEKaWYgX19uYW1lX189PSdfX21haW5fXyc6CiBwPWFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKGRlc2NyaXB0aW9uPV9fZG9jX18pO3AuYWRkX2FyZ3VtZW50KCctLWNhY2hlJyxyZXF1aXJlZD1UcnVlKTtwLmFkZF9hcmd1bWVudCgnLS1yZXBvJyxyZXF1aXJlZD1UcnVlKTtwLmFkZF9hcmd1bWVudCgnLS1vdXQnLHJlcXVpcmVkPVRydWUpO2E9cC5wYXJzZV9hcmdzKCk7bWFuaWZlc3QsZGF0YT1idWlsZChhLmNhY2hlLGEucmVwbyk7b3V0PVBhdGgoYS5vdXQpCiB3aXRoIG91dC5vcGVuKCd4JykgYXMgZjpqc29uLmR1bXAobWFuaWZlc3QsZixpbmRlbnQ9Mik7Zi53cml0ZSgnXG4nKQogcHJpbnQoanNvbi5kdW1wcyh7J21vZGVsX2NhY2hlX21hbmlmZXN0JzpzdHIob3V0KSwnbW9kZWxfY2FjaGVfbWFuaWZlc3Rfc2hhMjU2JzpzaGEob3V0KSwnZGF0YV9maWxlcyc6ZGF0YX0saW5kZW50PTIpKQo=', 'official_manifest.json': 'ewogICJoYXJuZXNzL190cmFpbi5weSI6ICI0NWE2MjY0MmRjZWFiMzJjZjNlZmNkYTUwNGRlODdhOWMyNWVhMjhiMTE5YjE4MDZmNTZkMTRmZjExNDZhODlkIiwKICAiaGFybmVzcy9jaGVja19ydWxlcy5weSI6ICI4NzlhYjE4NjE2NzRmNmU3MTRlYWRkMDBhYWYwNDQyMzVkOTg1MjZkZDc1ZmFiMTAzNWRlZWMzNDAxMTIxZWY1IiwKICAiaGFybmVzcy9jb21tb24ucHkiOiAiYzgzMDY2ZjI0ZjI2NmRhYjYyZWJjYzljMmQ0NGY1YTY1OTAyNzYwYjUzZGE5MTJlNjVlYmZlNTExNTc5YzJkYiIsCiAgImhhcm5lc3MvZXZhbF9hZGFwdGVyLnB5IjogIjQ3Y2RkYmI4ZTQ1NzljNjlmOTUzNWIwOTlmZjIzNzFiMWMzYWYwOGJmYmQ5M2QxNGJmNzlhYWRkNzRkMzg0ODIiLAogICJoYXJuZXNzL2V4YW1wbGVfcmVmZXJlbmNlLnB5IjogIjZhMTEwNTVmNGExODIyZTUzOTcxYzIyOGM2ZTNlM2FhNmI4OGVkZDllYjZkYjgxYTM4OGUyNmE3MGEwMzBjZWIiLAogICJoYXJuZXNzL3J1bl90YXNrLnB5IjogImQ5MzVhODk1OWYzNDg5NzczMTNmMmVlZjM2ODcwZmYyNjk5YjliYTAxM2E3OTc4OTE4ZmYzNWUxNDg5NjhhNzkiLAogICJoYXJuZXNzL3Njb3JlLnB5IjogIjczMWM5YjY0NGRjYzQwNjliMGIwNjk2MTZmNDM4MDY0ZDFmN2YzNzZhMjAyZmViMTUyMDFmZDkxMGVjNjg4YzUiLAogICJ0YXNrL3ByZXBhcmVfZGF0YS5weSI6ICJhODU0M2YxMWE1NWFmZjgwZGJkYzcxNGEwZTYyMzkxMjQ0YTUyOTY2MTZlZWEyZTM0ZGMyZWIzZmYzZWMxOGY3IiwKICAidGFzay90YXNrLnlhbWwiOiAiZGQ3MGM3MzlmY2QxNDNlMDZhZWYzYmIxZWZjYzkzODZmNjgyYmYxZGUyNmYzMGUwODQwOWQ4NmE5N2ZmNDdjZSIsCiAgInRlc3RzL2NvbmZ0ZXN0LnB5IjogImE4MmVhYmFiNDY4ZWEwZWYxZTAwM2NjY2Q3YzM4MjdjZTJmNGQwMGQwOWMzMjk0OGEzNGZiNDgyMjFmNzMzN2EiLAogICJ0ZXN0cy90ZXN0X2NhbGxiYWNrLnB5IjogIjE5MTkzMDNhMzY3MGMwYTMzYWJiZmNlOWIzM2VhYjA5MTNiYjY0YTNhMjFjMzMxMmQ4NjUzMWMzMmI1YmY0OTAiLAogICJ0ZXN0cy90ZXN0X2NvbmZpZy5weSI6ICJkYjgwOGM0NDQ3MGQxNDRiN2NmMThiNzMyOTQ5MmY0MWY3NmUzN2NmYzI0MzA2ZTdjYmEwNDg4YmNhOWQ3ODdjIiwKICAidGVzdHMvdGVzdF9mb3JtYXRzLnB5IjogIjVkNjU2ZWQ1MGM1OWY3OTRlOTM0NDczZDQ1OWVhODU2MzVmODkyYmEwMjhlMTkyNDE0NDc0ZTliNWQ2NTkxOWUiLAogICJ0ZXN0cy90ZXN0X2xvYWRlci5weSI6ICIzZGFkMWJlYTJkMzNhYWI5N2U1MmI3Y2Q0NDg4OGQ3NjhiYzk1MTlhZjQ0ZmJlNmNmM2U5NmMyNjAwMjY2MzJmIiwKICAidGVzdHMvdGVzdF90cmFpbmVyX2luaXQucHkiOiAiYjc2MDhkNjJhNDA5ODNiZjdjOWU1YjI3MGUwZTUyN2I2NDFhODM3ZTQ2MmQ2ZGYyOTA5ZmVjODhmNjA3ZTgyMSIKfQo=', 'runtime_api_reference.json': 'ewogICJwcm92ZW5hbmNlIjogIlBhcmVudCBhZ2VudCBhdXRoZW50aWNhdGVkIHJlYWQtb25seSBydW50aW1lIEdFVCwgdmVyaWZpZWQgVExTLCAyMDI2LTEwLTMwIGFyb3VuZCAyMDo1MCBVVEM7IG5vIHJlcXVlc3QgYnkgdGhpcyBhZGFwdGVyIGF1dGhvciBhbmQgbm8gR1BVIGpvYiIsCiAgImltYWdlIjogInNoYTI1NjpkMjgxMzlmNjljNTZhYzg5N2IyNjQ4N2YwZjdkODIzNmQ3MDFmMDMwMjE0MzQxYjUxMTNiNDcxNDI2N2MzZWM5IiwKICAiZ3B1IjogIlRlc2xhIFQ0IiwgInZyYW1fbWliIjogMTUzNjAsICJkcml2ZXIiOiAiNTgwLjE3OC4wNCIsCiAgInB5dGhvbiI6ICIzLjEzLjE1IiwgInRvcmNoIjogIjIuMTEuMCtjdTEyOCIsICJjdWRhX2J1aWxkIjogIjEyLjgiLCAidHJpdG9uIjogIjMuNi4wIiwgIm51bXB5IjogIjIuMy41IiwKICAiY3B1X2FmZmluaXR5IjogWzAsMSwyLDNdLCAiY2dyb3VwX3JhbV9naWIiOiAzMCwgImhvc3RfcmVwb3J0ZWRfcmFtX2dpYiI6IDI5LjM3LAogICJTT1VQX1JFUE8iOiAiL2thZ2dsZS93b3JraW5nL3JlcG8iLCAiU09VUF9SRVNVTFRTIjogIi9rYWdnbGUvd29ya2luZy9yZXN1bHRzIiwKICAib2ZmaWNpYWxfY29tbWl0IjogImM3YzJmY2QxNTk3MmNlOGI4OThkOGQzZmY3MzJlZTE5MmM4MTdjZWIiLAogICJ0cmFpbmluZ190aW1lb3V0X3MiOiA4MCwgInB1YmxpY19xdWFsaXR5X2RlbHRhX2xpbWl0X3BjdCI6IDAuNSwgInBlYWtfcmVncmVzc2lvbl9saW1pdF9wY3QiOiAxMCwKICAiY29uY3VycmVuY3kiOiAxLCAibmV0d29yayI6ICJub25lIiwgImF1dGhlbnRpY2F0ZWRfdXBsb2FkX2F1dG9fc3RhcnRzIjogdHJ1ZSwKICAibm90ZWJvb2tfbGltaXRfYnl0ZXMiOiA4Mzg4NjA4LCAiZXhwb3J0X2xpbWl0X2J5dGVzIjogNjcxMDg4NjQsCiAgImRlZmF1bHRfam9iX3RpbWVvdXRfcyI6IDM2MDAsICJtYXhfam9iX3RpbWVvdXRfcyI6IDE0NDAwLAogICJkZXZpY2VfZXhlY3V0aW9uX3Byb29mIjogZmFsc2UsICJncHVfam9iX3J1bl9ieV91cyI6IGZhbHNlCn0K', 'web_runner.py': 'IiIiQWN0dWFsLWJhdGNoIG5hdGl2ZS1yZXR1cm4gTUxQIHNoYWRvdyBvbmx5OyBubyBvcHRpbWl6ZXIsIHRyYWluIGxvb3AsIGJhc2VsaW5lIHJlcGxheSBvciBzY29yZS4iIiIKaW1wb3J0IGFyZ3BhcnNlLGZjbnRsLGhhc2hsaWIsanNvbixvcyxzaHV0aWwsc3lzLHRocmVhZGluZyx0aW1lLHV1aWQsemlwZmlsZQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKaW1wb3J0IHByb2Nlc3NfZ3VhcmQKaW1wb3J0IG93bmVkX3Byb2Nlc3NfdHJlZSBhcyB0cmVlCkNPTU1JVD0nYzdjMmZjZDE1OTcyY2U4Yjg5OGQ4ZDNmZjczMmVlMTkyYzgxN2NlYicKTU9ERT0nYWN0dWFsX2JhdGNoX25hdGl2ZV9yZXR1cm5fbWxwX3NoYWRvd192MScKU1RBVFVTPSdDT01QTEVURV9OQVRJVkVfUkVUVVJOX01MUF9TSEFET1dfRElBR05PU1RJQycKTE9DSz1QYXRoKCcvdG1wL3NvdXAtZ3B1MC5sb2NrJyk7TUFSS0VSPVBhdGgoJy90bXAvc291cC1ncHUwLmxvY2suYmxvY2tlZCcpCkVYUE9SVF9CVURHRVQ9NTYqMTAyNCoqMgpFWFBPUlRfT01JVFRFRD17fQpkZWYgc2hhKHApOnJldHVybiBoYXNobGliLnNoYTI1NihQYXRoKHApLnJlYWRfYnl0ZXMoKSkuaGV4ZGlnZXN0KCkKCgpkZWYgcmVhZChwKTpyZXR1cm4ganNvbi5sb2FkcyhQYXRoKHApLnJlYWRfdGV4dCgpKQoKCmRlZiB3cml0ZShwLHZhbHVlKToKIHA9UGF0aChwKTtwZW5kaW5nPXAud2l0aF9uYW1lKHAubmFtZSsnLnBlbmRpbmcnKTtwZW5kaW5nLndyaXRlX3RleHQoanNvbi5kdW1wcyh2YWx1ZSxpbmRlbnQ9MixhbGxvd19uYW49RmFsc2UpKydcbicpO3BlbmRpbmcucmVwbGFjZShwKQoKCmRlZiB2ZXJpZnlfb2ZmaWNpYWwocmVwbyxtYW5pZmVzdCk6CiBpZiBub3QgcmVwby5pc19kaXIoKSBvciByZXBvLmlzX3N5bWxpbmsoKTpyYWlzZSBSdW50aW1lRXJyb3IoJ1NPVVBfUkVQTyBtdXN0IGJlIGEgcmVhbCBwcmVwYXJlZCBqb2IgZGlyZWN0b3J5JykKIGZvciBuYW1lLHdhbnRlZCBpbiBtYW5pZmVzdC5pdGVtcygpOgogIHA9cmVwby9uYW1lCiAgaWYgbm90IHAuaXNfZmlsZSgpIG9yIHAuaXNfc3ltbGluaygpIG9yIHNoYShwKSE9d2FudGVkOnJhaXNlIFJ1bnRpbWVFcnJvcignUHJvdGVjdGVkIG9mZmljaWFsIHNvdXJjZSBtaXNtYXRjaDogJytuYW1lKQogYWN0dWFsPXtzdHIocC5yZWxhdGl2ZV90byhyZXBvKSkgZm9yIGZvbGRlciBpbiAoJ2hhcm5lc3MnLCd0YXNrJywndGVzdHMnKSBmb3IgcCBpbiAocmVwby9mb2xkZXIpLnJnbG9iKCcqJykKICAgICAgICAgaWYgcC5pc19maWxlKCkgYW5kICdfX3B5Y2FjaGVfXycgbm90IGluIHAucGFydHMgYW5kIHAuc3VmZml4IG5vdCBpbiAoJy5weWMnLCcucHlvJyl9CiBpZiBhY3R1YWwhPXNldChtYW5pZmVzdCk6cmFpc2UgUnVudGltZUVycm9yKCdVbmV4cGVjdGVkIHByb3RlY3RlZCBvZmZpY2lhbCBmaWxlcycpCgoKZGVmIGV4cG9ydF9zbWFsbChydW4sZXhwb3J0KToKICIiIlJlYWQtb25seSBpbmNyZW1lbnRhbCBleHBvcnQ7IG5vIGFkYXB0ZXJzLCB3ZWlnaHRzIG9yIGNvbXBpbGVyIGNhY2hlcy4iIiIKIHVzZWQ9c3VtKHAuc3RhdCgpLnN0X3NpemUgZm9yIHAgaW4gZXhwb3J0LnJnbG9iKCcqJykgaWYgcC5pc19maWxlKCkpCiBleGNsdWRlZD17J2FkYXB0ZXInLCd0cml0b25fY2FjaGUnLCdpbmR1Y3Rvcl9jYWNoZScsJ3ByZWZsaWdodF90cml0b24nLCdwcmVmbGlnaHRfaW5kdWN0b3InLCdvcmlnaW5hbF9zb2x1dGlvbid9CiBmb3IgYmFzZSxkaXJzLGZpbGVzIGluIG9zLndhbGsocnVuKToKICBkaXJzWzpdPVtkIGZvciBkIGluIGRpcnMgaWYgZCBub3QgaW4gZXhjbHVkZWQgYW5kIG5vdCBkLnN0YXJ0c3dpdGgoJ2NoZWNrcG9pbnQtJykgYW5kIG5vdCAoUGF0aChiYXNlKS9kKS5pc19zeW1saW5rKCldCiAgZm9yIG5hbWUgaW4gZmlsZXM6CiAgIHBhdGg9UGF0aChiYXNlKS9uYW1lO3JlbD1wYXRoLnJlbGF0aXZlX3RvKHJ1bikKICAgdHJ5OgogICAgaWYgbm90IHBhdGguaXNfc3ltbGluaygpIGFuZCBwYXRoLnN1ZmZpeCBpbiAoJy5qc29uJywnLmpzb25sJywnLmxvZycsJy55YW1sJyk6CiAgICAgaWYgcGF0aC5zdGF0KCkuc3Rfc2l6ZT49NCoxMDI0KioyOgogICAgICBFWFBPUlRfT01JVFRFRFtzdHIocmVsKV09J3Blcl9maWxlX2xpbWl0XzRNaUInO2NvbnRpbnVlCiAgICAgZGVzdD1leHBvcnQvJ3JhdycvcmVsO2Rlc3QucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSxleGlzdF9vaz1UcnVlKQogICAgIGlmIG5vdCBkZXN0LmV4aXN0cygpIGFuZCBzdW0oMSBmb3IgcCBpbiBleHBvcnQucmdsb2IoJyonKSBpZiBwLmlzX2ZpbGUoKSk+PTI0OgogICAgICBFWFBPUlRfT01JVFRFRFtzdHIocmVsKV09J3BlcnNpc3RlbnRfZmlsZV9saW1pdF8yNCc7Y29udGludWUKICAgICBwcmlvcj1kZXN0LnN0YXQoKS5zdF9zaXplIGlmIGRlc3QuZXhpc3RzKCkgZWxzZSAwCiAgICAgc2l6ZT1wYXRoLnN0YXQoKS5zdF9zaXplCiAgICAgaWYgdXNlZC1wcmlvcitzaXplPkVYUE9SVF9CVURHRVQ6CiAgICAgIEVYUE9SVF9PTUlUVEVEW3N0cihyZWwpXT0nYWdncmVnYXRlX2J1ZGdldF81Nk1pQic7Y29udGludWUKICAgICB0ZW1wPWRlc3Qud2l0aF9uYW1lKGRlc3QubmFtZSsnLmNvcHlpbmcnKTtzaHV0aWwuY29weWZpbGUocGF0aCx0ZW1wKTt0ZW1wLnJlcGxhY2UoZGVzdCk7dXNlZD11c2VkLXByaW9yK3NpemUKICAgZXhjZXB0IChGaWxlTm90Rm91bmRFcnJvcixPU0Vycm9yKTpwYXNzCgoKY2xhc3MgRXhwb3J0ZXI6CiBkZWYgX19pbml0X18oc2VsZixydW4sZXhwb3J0KTpzZWxmLnJ1bj1ydW47c2VsZi5leHBvcnQ9ZXhwb3J0O3NlbGYuc3RvcD10aHJlYWRpbmcuRXZlbnQoKTtzZWxmLnRocmVhZD1Ob25lCiBkZWYgX19lbnRlcl9fKHNlbGYpOgogIGRlZiBsb29wKCk6CiAgIHdoaWxlIG5vdCBzZWxmLnN0b3Aud2FpdCgxKTpleHBvcnRfc21hbGwoc2VsZi5ydW4sc2VsZi5leHBvcnQpCiAgc2VsZi50aHJlYWQ9dGhyZWFkaW5nLlRocmVhZCh0YXJnZXQ9bG9vcCxkYWVtb249VHJ1ZSk7c2VsZi50aHJlYWQuc3RhcnQoKTtyZXR1cm4gc2VsZgogZGVmIF9fZXhpdF9fKHNlbGYsKmV4Yyk6c2VsZi5zdG9wLnNldCgpO3NlbGYudGhyZWFkLmpvaW4oKTtleHBvcnRfc21hbGwoc2VsZi5ydW4sc2VsZi5leHBvcnQpCgoKZGVmIGlkbGVfY2hlY2soZW52LHJlcG8scnVuLGxvY2tmZCk6CiBjb2RlPScnJ2ltcG9ydCBqc29uLHN1YnByb2Nlc3Msc3lzCnA9c3VicHJvY2Vzcy5ydW4oWydudmlkaWEtc21pJywnLS1xdWVyeS1jb21wdXRlLWFwcHM9cGlkLHVzZWRfZ3B1X21lbW9yeScsJy0tZm9ybWF0PWNzdixub2hlYWRlcixub3VuaXRzJ10sY2FwdHVyZV9vdXRwdXQ9VHJ1ZSx0ZXh0PVRydWUsY2hlY2s9VHJ1ZSkKYXNzZXJ0IG5vdCBwLnN0ZG91dC5zdHJpcCgpLCdHUFUgcHJvY2Vzc2VzIHN0aWxsIGFjdGl2ZSBhZnRlciBwcm9iZS9wcmVmbGlnaHQnCnI9c3VicHJvY2Vzcy5ydW4oWydudmlkaWEtc21pJywnLS1xdWVyeS1ncHU9aW5kZXgsbWVtb3J5LnVzZWQnLCctLWZvcm1hdD1jc3Ysbm9oZWFkZXIsbm91bml0cyddLGNhcHR1cmVfb3V0cHV0PVRydWUsdGV4dD1UcnVlLGNoZWNrPVRydWUpCnJvd3M9W2xpbmUuc3BsaXQoJywnKSBmb3IgbGluZSBpbiByLnN0ZG91dC5zcGxpdGxpbmVzKCkgaWYgbGluZS5zdHJpcCgpXQphc3NlcnQgcm93cyBhbmQgc3VtKGludCh4WzFdKSBmb3IgeCBpbiByb3dzKTwxMDAwLCdHUFUgcGh5c2ljYWwgaW52ZW50b3J5IG5vdCBpZGxlJwpwcmludChqc29uLmR1bXBzKHsnaWRsZSc6VHJ1ZSwncGh5c2ljYWxfbWVtb3J5X3Jvd3MnOnJvd3N9KSknJycKIHJldHVybiBwcm9jZXNzX2d1YXJkLnJ1bihbc3lzLmV4ZWN1dGFibGUsJy1jJyxjb2RlXSxjd2Q9cmVwbyxlbnY9ZW52LGxvZz1ydW4vJ2lkbGUubG9nJyx0aW1lb3V0PTIwLGxvY2tfZmQ9bG9ja2ZkKQoKCmRlZiBtYWluKCk6CiBwPWFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKCk7cC5hZGRfYXJndW1lbnQoJy0tbW9kZScsY2hvaWNlcz1bTU9ERV0scmVxdWlyZWQ9VHJ1ZSk7YT1wLnBhcnNlX2FyZ3MoKQogaGVyZT1QYXRoKF9fZmlsZV9fKS5yZXNvbHZlKCkucGFyZW50CiBpZiBub3Qgb3MuZW52aXJvbi5nZXQoJ1NPVVBfUkVQTycpIG9yIG5vdCBvcy5lbnZpcm9uLmdldCgnU09VUF9SRVNVTFRTJyk6cmFpc2UgU3lzdGVtRXhpdCgnUHJlcGFyZWQgZW52aXJvbm1lbnQgbWlzc2luZzsgbm8gbWFudWFsL25ldHdvcmsgZmFsbGJhY2snKQogcmVwbz1QYXRoKG9zLmVudmlyb25bJ1NPVVBfUkVQTyddKS5hYnNvbHV0ZSgpO3Jlc3VsdHM9UGF0aChvcy5lbnZpcm9uWydTT1VQX1JFU1VMVFMnXSkuYWJzb2x1dGUoKQogaWYgcmVwby5pc19zeW1saW5rKCkgb3IgcmVzdWx0cy5pc19zeW1saW5rKCkgb3IgcmVwbz09cmVzdWx0czpyYWlzZSBTeXN0ZW1FeGl0KCdSZWFsIGRpc3RpbmN0IHByZXBhcmVkIGRpcmVjdG9yaWVzIHJlcXVpcmVkJykKIGlmIG5vdCBQYXRoKCcva2FnZ2xlL3dvcmtpbmcnKS5pc19kaXIoKTpyYWlzZSBTeXN0ZW1FeGl0KCdQcmVwYXJlZCB3b3JraW5nIGRpcmVjdG9yeSBtaXNzaW5nJykKIHJlc3VsdHMubWtkaXIocGFyZW50cz1UcnVlLGV4aXN0X29rPVRydWUpO2lkZW50aXR5PU1PREUrJy0nK3V1aWQudXVpZDQoKS5oZXhbOjEyXQogZXhwb3J0PXJlc3VsdHMvaWRlbnRpdHk7ZXhwb3J0Lm1rZGlyKGV4aXN0X29rPUZhbHNlKTtydW49UGF0aCgnL2thZ2dsZS93b3JraW5nJykvaWRlbnRpdHk7cnVuLm1rZGlyKGV4aXN0X29rPUZhbHNlKQogc3VtbWFyeT17J3N0YXR1cyc6J1BSRVBBUklOR19OQVRJVkVfT05FX1NURVBfRElBR05PU1RJQycsJ3Njb3BlJzonQUNUVUFMX0JBVENIX05BVElWRV9SRVRVUk5fTUxQX1NIQURPV19ESUFHTk9TVElDX09OTFknLAogICdtb2RlJzpNT0RFLCdzb3VyY2VfY29tbWl0X3JlZmVyZW5jZSc6Q09NTUlULCdjb21wbGV0ZV9maXhlZF90YXNrJzpGYWxzZSwnY29tcGxldGVfZGlhZ25vc3RpYyc6RmFsc2UsCiAgJ29wdGltaXplcl9zdGVwcyc6MCwndHJhaW5pbmdfc3RlcF9jYWxscyc6Tm9uZSwnc2NvcmUnOk5vbmUsJ2FkbWlzc2lvbic6J1VOS05PV04nLAogICd3b3JrZXJfdGltZW91dF9zJzo4MCwnZXBoZW1lcmFsX2RpYWdub3N0aWNfb3duZXJfaW5fY2hpbGQnOlRydWUsJ3NvbHV0aW9uX2luc3RhbGxlZCc6RmFsc2UsJ3RyYWluaW5nX2xvb3BfaW52b2tlZCc6RmFsc2UsJ3B1YmxpY19ldmFsdWF0aW9uX2ludm9rZWQnOkZhbHNlfQogd3JpdGUoZXhwb3J0LydzdGF0dXMuanNvbicsc3VtbWFyeSk7c3RhcnRlZD10aW1lLm1vbm90b25pYygpO2xvY2tmZD1Ob25lCiB0cnk6CiAgaWYgTUFSS0VSLmV4aXN0cygpOnJhaXNlIFJ1bnRpbWVFcnJvcignR1BVIHF1YXJhbnRpbmUgcmVxdWlyZXMgb3BlcmF0b3IgcmVjb25jaWxpYXRpb24nKQogIGxvY2tmZD1vcy5vcGVuKExPQ0ssb3MuT19DUkVBVHxvcy5PX1JEV1J8b3MuT19OT0ZPTExPVywwbzYwMCk7ZmNudGwuZmxvY2sobG9ja2ZkLGZjbnRsLkxPQ0tfRVh8ZmNudGwuTE9DS19OQikKICBpZiBNQVJLRVIuZXhpc3RzKCk6cmFpc2UgUnVudGltZUVycm9yKCdHUFUgcXVhcmFudGluZSBkZXRlY3RlZCBhZnRlciBsb2NrJykKICB0cmVlLnByZXBhcmUoKTt0cmVlLmJlZ2luX3F1YXJhbnRpbmUoTUFSS0VSKQogIG9mZmljaWFsPXJlYWQoaGVyZS8nb2ZmaWNpYWxfbWFuaWZlc3QuanNvbicpO3ZlcmlmeV9vZmZpY2lhbChyZXBvLG9mZmljaWFsKQogIGJ1bmRsZT1yZWFkKGhlcmUvJ3dvcmtlcl9tYW5pZmVzdC5qc29uJyk7c291cmNlcz1idW5kbGVbJ3NvdXJjZXMnXQogIGlmIHNldChzb3VyY2VzKSE9eydiYXRjaF9iaW5kaW5nLmpzb24nLCdiYXRjaF9oYXNoZXMuanNvbicsJ25hdGl2ZV9zb3VyY2VfcGlucy5qc29uJywncHJvZmlsZXJfaGVscGVycy5weScsJ3NvbHV0aW9uL19faW5pdF9fLnB5Jywnc29sdXRpb24vYmFja3dhcmRfcGhhc2UucHknLCdzb2x1dGlvbi9jb21wYWN0X21scF9pbnN0YWxsLnB5Jywnc29sdXRpb24vZm9yd2FyZF9vYnNlcnZlci5weScsJ3NvbHV0aW9uL21scF9oYWxmL19faW5pdF9fLnB5Jywnc29sdXRpb24vbWxwX2hhbGYvbmF0aXZlX293bmVyLnB5Jywnc29sdXRpb24vbWxwX2hhbGYvb3duZWRfaGFsZi5weScsJ3NvbHV0aW9uL21scF9oYWxmL3N0YXJ0ZXJfaGFsZi5weScsJ3NvbHV0aW9uL21scF9oYWxmL3N0YXJ0ZXJfc3dpZ2x1LnB5Jywnc29sdXRpb24vbWxwX29ic2VydmF0aW9uX3Nlc3Npb24ucHknLCdzb2x1dGlvbi9yb3dfYWRqb2ludC5weScsJ3NvbHV0aW9uL3RyYWluZXJfc2hhZG93LnB5Jywnd29ya2VyLnB5J306cmFpc2UgUnVudGltZUVycm9yKCdFeGFjdCBkaWFnbm9zdGljIHNvdXJjZSBpbnZlbnRvcnkgcmVxdWlyZWQnKQogIGFyY2hpdmU9aGVyZS8nZGlhZ25vc3RpY19zb3VyY2VzLnppcCcKICBpZiBub3QgYXJjaGl2ZS5pc19maWxlKCkgb3IgYXJjaGl2ZS5pc19zeW1saW5rKCkgb3Igc2hhKGFyY2hpdmUpIT1idW5kbGVbJ3ppcF9zaGEyNTYnXTpyYWlzZSBSdW50aW1lRXJyb3IoJ0Zyb3plbiBzb3VyY2UgYXJjaGl2ZSBjaGFuZ2VkJykKICBzb3VyY2Vfcm9vdD1oZXJlLydkaWFnbm9zdGljX3NvdXJjZSc7c291cmNlX3Jvb3QubWtkaXIoZXhpc3Rfb2s9RmFsc2UpCiAgd2l0aCB6aXBmaWxlLlppcEZpbGUoYXJjaGl2ZSkgYXMgejoKICAgaW5mb3M9ei5pbmZvbGlzdCgpCiAgIGlmIGxlbihpbmZvcykhPWxlbihzb3VyY2VzKSBvciB7aS5maWxlbmFtZSBmb3IgaSBpbiBpbmZvc30hPXNldChzb3VyY2VzKTpyYWlzZSBSdW50aW1lRXJyb3IoJ0RpYWdub3N0aWMgc291cmNlIGFyY2hpdmUgaW52ZW50b3J5IG1pc21hdGNoJykKICAgaWYgc3VtKGkuZmlsZV9zaXplIGZvciBpIGluIGluZm9zKT44KjEwMjQqKjI6cmFpc2UgUnVudGltZUVycm9yKCdEaWFnbm9zdGljIHNvdXJjZSBhcmNoaXZlIGV4Y2VlZHMgYm91bmQnKQogICBmb3IgaW5mbyBpbiBpbmZvczoKICAgIGlmIGluZm8uZmlsZV9zaXplPjEwMjQqKjIgb3IgKChpbmZvLmV4dGVybmFsX2F0dHI+PjE2KSYwbzE3MDAwMCk9PTBvMTIwMDAwOnJhaXNlIFJ1bnRpbWVFcnJvcignVW5zdXBwb3J0ZWQgc291cmNlIGFyY2hpdmUgbWVtYmVyJykKICAgIHJhdz16LnJlYWQoaW5mbykKICAgIGlmIGhhc2hsaWIuc2hhMjU2KHJhdykuaGV4ZGlnZXN0KCkhPXNvdXJjZXNbaW5mby5maWxlbmFtZV06cmFpc2UgUnVudGltZUVycm9yKCdGcm96ZW4gc291cmNlIGJ5dGVzIGNoYW5nZWQ6ICcraW5mby5maWxlbmFtZSkKICAgIHRhcmdldD1zb3VyY2Vfcm9vdC9pbmZvLmZpbGVuYW1lO3RhcmdldC5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLGV4aXN0X29rPVRydWUpO3RhcmdldC53cml0ZV9ieXRlcyhyYXcpCiAgd29ya2VyPXNvdXJjZV9yb290Lyd3b3JrZXIucHknCiAgd3JpdGUocnVuLydkaWFnbm9zdGljX3NvdXJjZV9tYW5pZmVzdC5qc29uJyxzb3VyY2VzKTtzdW1tYXJ5Wyd3b3JrZXJfc2hhMjU2J109c291cmNlc1snd29ya2VyLnB5J10KICBlbnY9ZGljdChvcy5lbnZpcm9uLEhGX0hVQl9PRkZMSU5FPScxJyxIRl9EQVRBU0VUU19PRkZMSU5FPScxJyxUUkFOU0ZPUk1FUlNfT0ZGTElORT0nMScsUFlUSE9ORE9OVFdSSVRFQllURUNPREU9JzEnLFBZVEhPTlVOQlVGRkVSRUQ9JzEnKQogIGVudi5wb3AoJ1NPVVBfUk9UQVJZX0NPREVHRU5fRElSJyxOb25lKQogIHdpdGggRXhwb3J0ZXIocnVuLGV4cG9ydCk6CiAgIHByb2JlPXByb2Nlc3NfZ3VhcmQucnVuKFtzeXMuZXhlY3V0YWJsZSxzdHIoaGVyZS8ncnVudGltZV9wcm9iZS5weScpLHN0cihydW4vJ3J1bnRpbWUuanNvbicpXSxjd2Q9cmVwbyxlbnY9ZW52LGxvZz1ydW4vJ3J1bnRpbWUubG9nJyx0aW1lb3V0PTQwLGxvY2tfZmQ9bG9ja2ZkKQogICBpZiBwcm9iZVsnZXhpdF9jb2RlJ10gb3Igbm90IHJlYWQocnVuLydydW50aW1lLmpzb24nKS5nZXQoJ3Bhc3NlZCcpOnJhaXNlIFJ1bnRpbWVFcnJvcignUnVudGltZSBnYXRlIGZhaWxlZCcpCiAgIHJ1bnRpbWU9cmVhZChydW4vJ3J1bnRpbWUuanNvbicpO3N1bW1hcnlbJ3J1bnRpbWUnXT1ydW50aW1lCiAgIGZyb20gcHJlcGFyZV9hc3NldF9tYW5pZmVzdCBpbXBvcnQgYnVpbGQKICAgYXNzZXRzLGRhdGE9YnVpbGQocnVudGltZVsncmVzb2x2ZWRfcHVibGljX2h1Yl9jYWNoZSddLHJlcG8pCiAgIHdyaXRlKHJ1bi8nbW9kZWxfY2FjaGVfbWFuaWZlc3QuanNvbicsYXNzZXRzKTt3cml0ZShydW4vJ3B1YmxpY19kYXRhX21hbmlmZXN0Lmpzb24nLGRhdGEpCiAgIGJlZm9yZT1ydW4vJ2lkbGVfYmVmb3JlJztiZWZvcmUubWtkaXIoKQogICBpZGxlPWlkbGVfY2hlY2soZW52LHJlcG8sYmVmb3JlLGxvY2tmZCkKICAgaWYgaWRsZVsnZXhpdF9jb2RlJ106cmFpc2UgUnVudGltZUVycm9yKCdQcmUtd29ya2VyIEdQVSBpZGxlIGdhdGUgZmFpbGVkJykKICAgd29ya2VyX2Vudj1kaWN0KGVudixUUklUT05fQ0FDSEVfRElSPXN0cihydW4vJ3ByZWZsaWdodF90cml0b24nKSxUT1JDSElORFVDVE9SX0NBQ0hFX0RJUj1zdHIocnVuLydwcmVmbGlnaHRfaW5kdWN0b3InKSkKICAgcHJvY2Vzcz1wcm9jZXNzX2d1YXJkLnJ1bihbc3lzLmV4ZWN1dGFibGUsJy1CJyxzdHIod29ya2VyKSwnLS1vdXQnLHN0cihydW4vJ3Byb2ZpbGUuanNvbicpLCctLXJlcG8nLHN0cihyZXBvKV0sY3dkPXJlcG8sZW52PXdvcmtlcl9lbnYsbG9nPXJ1bi8nd29ya2VyLmxvZycsdGltZW91dD04MCxsb2NrX2ZkPWxvY2tmZCkKICAgc3VtbWFyeVsnZGlhZ25vc3RpY19wcm9jZXNzJ109cHJvY2VzcwogICBpZiBwcm9jZXNzWydleGl0X2NvZGUnXSBvciBwcm9jZXNzLmdldCgndGltZWRfb3V0Jykgb3IgcHJvY2Vzcy5nZXQoJ2ludGVycnVwdGVkJykgb3IgcHJvY2Vzcy5nZXQoJ2NsZWFudXBfY29uZmlybWVkJykgaXMgbm90IFRydWU6CiAgICByYWlzZSBSdW50aW1lRXJyb3IoJ09uZS1zdGVwIGRpYWdub3N0aWMgcHJvY2VzcyBpbmNvbXBsZXRlIG9yIGNlbnNvcmVkJykKICAgcmVwb3J0PXJlYWQocnVuLydwcm9maWxlLmpzb24nKQogICBpZiAocmVwb3J0LmdldCgnc3RhdHVzJykhPVNUQVRVUyBvciByZXBvcnQuZ2V0KCdzY29wZScpIT0nQUNUVUFMX0JBVENIX05BVElWRV9SRVRVUk5fTUxQX1NIQURPV19ESUFHTk9TVElDX09OTFknCiAgICAgICBvciByZXBvcnQuZ2V0KCdjb21wbGV0ZV9maXhlZF90YXNrJykgaXMgbm90IEZhbHNlIG9yIHJlcG9ydC5nZXQoJ3Njb3JlJykgaXMgbm90IE5vbmUgb3IgcmVwb3J0LmdldCgnYWRtaXNzaW9uJykhPSdVTktOT1dOJwogICAgICAgb3IgdHlwZShyZXBvcnQuZ2V0KCdvcHRpbWl6ZXJfc3RlcHMnKSkgaXMgbm90IGludCBvciByZXBvcnRbJ29wdGltaXplcl9zdGVwcyddIT0wCiAgICAgICBvciB0eXBlKHJlcG9ydC5nZXQoJ3RyYWluaW5nX3N0ZXBfY2FsbHMnKSkgaXMgbm90IGludCBvciByZXBvcnRbJ3RyYWluaW5nX3N0ZXBfY2FsbHMnXSE9MQogICAgICAgb3IgcmVwb3J0LmdldCgnbmF0aXZlX2JhY2t3YXJkX3Jlc3VsdF9yZXR1cm5lZCcpIGlzIG5vdCBUcnVlCiAgICAgICBvciByZXBvcnQuZ2V0KCdyYW5nZV9jZXJ0aWZpY2F0ZScpIGlzIG5vdCBGYWxzZSBvciByZXBvcnQuZ2V0KCdwcm9kdWN0aW9uX2NsZWFyYW5jZScpIGlzIG5vdCBGYWxzZSk6CiAgICByYWlzZSBSdW50aW1lRXJyb3IoJ09uZS1zdGVwIGRpYWdub3N0aWMgc2NvcGUvc3RhdHVzIGNvbnRyYWN0IGZhaWxlZCcpCiAgIGFmdGVyPXJ1bi8naWRsZV9hZnRlcic7YWZ0ZXIubWtkaXIoKQogICBpZGxlPWlkbGVfY2hlY2soZW52LHJlcG8sYWZ0ZXIsbG9ja2ZkKQogICBpZiBpZGxlWydleGl0X2NvZGUnXTpyYWlzZSBSdW50aW1lRXJyb3IoJ1Bvc3Qtd29ya2VyIEdQVSBpZGxlIGdhdGUgZmFpbGVkJykKICAgdmVyaWZ5X29mZmljaWFsKHJlcG8sb2ZmaWNpYWwpCiAgIHN1bW1hcnkudXBkYXRlKHN0YXR1cz1TVEFUVVMsY29tcGxldGVfZGlhZ25vc3RpYz1UcnVlLHRyYWluaW5nX3N0ZXBfY2FsbHM9MSxkaWFnbm9zdGljX3JlcG9ydD0ncmF3L3Byb2ZpbGUuanNvbicpCiBleGNlcHQgQmFzZUV4Y2VwdGlvbiBhcyBlcnJvcjoKICBzdW1tYXJ5LnVwZGF0ZShzdGF0dXM9J0JMT0NLRURfRkFJTEVEX09SX0NFTlNPUkVEX05BVElWRV9ESUFHTk9TVElDJyxjb21wbGV0ZV9kaWFnbm9zdGljPUZhbHNlLGVycm9yPXJlcHIoZXJyb3IpKQogIGlmIGxvY2tmZCBpcyBub3QgTm9uZTp0cmVlLnF1YXJhbnRpbmUoJ05hdGl2ZSBkaWFnbm9zdGljIHJ1bm5lciBmYWlsdXJlOiAnK3JlcHIoZXJyb3IpKQogZmluYWxseToKICBleHBvcnRfc21hbGwocnVuLGV4cG9ydCkKICBpZiBFWFBPUlRfT01JVFRFRDoKICAgc3VtbWFyeS51cGRhdGUoc3RhdHVzPSdCTE9DS0VEX0ZBSUxFRF9PUl9DRU5TT1JFRF9OQVRJVkVfRElBR05PU1RJQycsY29tcGxldGVfZGlhZ25vc3RpYz1GYWxzZSxlcnJvcj0nUmVxdWlyZWQgYm91bmRlZCBleHBvcnQgaW5jb21wbGV0ZScpCiAgc3VtbWFyeS51cGRhdGUob3JjaGVzdHJhdG9yX3dhbGxfcz10aW1lLm1vbm90b25pYygpLXN0YXJ0ZWQsZXhwb3J0X29taXR0ZWQ9ZGljdChFWFBPUlRfT01JVFRFRCksCiAgICAgIG91dHB1dF9idWRnZXRfYnl0ZXM9RVhQT1JUX0JVREdFVCxwZXJzaXN0ZW50X2V4cG9ydF9maWxlX2xpbWl0PTI0LHNjb3JlPU5vbmUsYWRtaXNzaW9uPSdVTktOT1dOJyxjb21wbGV0ZV9maXhlZF90YXNrPUZhbHNlKQogIHdyaXRlKGV4cG9ydC8nc3RhdHVzLmpzb24nLHN1bW1hcnkpCiAgaWYgbG9ja2ZkIGlzIG5vdCBOb25lOnRyZWUuZmluaXNoX3F1YXJhbnRpbmUoKTtvcy5jbG9zZShsb2NrZmQpCiBwcmludChqc29uLmR1bXBzKHsnZXhwb3J0JzpzdHIoZXhwb3J0KSwnc3RhdHVzJzpzdW1tYXJ5WydzdGF0dXMnXSwnY29tcGxldGVfZGlhZ25vc3RpYyc6c3VtbWFyeVsnY29tcGxldGVfZGlhZ25vc3RpYyddLCdjb21wbGV0ZV9maXhlZF90YXNrJzpGYWxzZSwnc2NvcmUnOk5vbmUsJ2FkbWlzc2lvbic6J1VOS05PV04nfSksZmx1c2g9VHJ1ZSkKIHJhaXNlIFN5c3RlbUV4aXQoMCBpZiBzdW1tYXJ5Wydjb21wbGV0ZV9kaWFnbm9zdGljJ10gZWxzZSAxKQppZiBfX25hbWVfXz09J19fbWFpbl9fJzptYWluKCkK', 'diagnostic_sources.zip': '...', 'worker_manifest.json': 'ewogICJ6aXBfc2hhMjU2IjogIjlkOGQwNDNjMTQ3MzJiMGQ5NmUzMzViNGM2YTE2NTlkMmY0Y2Y2ZjAxZmMzZDI3NWJmNDliODUxMzc0M2I1YjYiLAogICJzb3VyY2VzIjogewogICAgImJhdGNoX2JpbmRpbmcuanNvbiI6ICIwNjU3MTVjNjY5ZjM4MzE5NWE5NzY3Y2UwZDhlZTVlNjdjN2ViZjYxY2I2ZDkxNjI2NDI5Njk4MmFkMjRkYTdiIiwKICAgICJiYXRjaF9oYXNoZXMuanNvbiI6ICJlMjZkNzFmMzMzZDJmN2IxZWI2ZGJiNzVhN2UyMjlmZTJiMTY0MjZhMWEyYzIzNGMxNDViYjI3ZjE4NzY1MDUwIiwKICAgICJuYXRpdmVfc291cmNlX3BpbnMuanNvbiI6ICI5MTQyNDRkNDYxZTRmYjc0ZmFiOTdiYmEyMDI4MTIzOTExNmZjOTdkOThlZWFmMDkwM2NlM2VhNDQ2ZDg3MGY1IiwKICAgICJwcm9maWxlcl9oZWxwZXJzLnB5IjogIjBhZWJhZjU0NmU4NDhlZWEzMzA5OTA2YzI5MTg5OTIyY2M2ZGI2NmExMTE0NTcxMTVjMjk5OTAxZTllZTg3ZTIiLAogICAgInNvbHV0aW9uL19faW5pdF9fLnB5IjogImMxMDNjZmFiYWRkNmVjOTdkMjc5OTVmMGQ4NTI3ZTM4ZjBhMmMyZmEzYWQ0NjE3ZGFiM2RlYTE3ODdiMGY1NzEiLAogICAgInNvbHV0aW9uL2JhY2t3YXJkX3BoYXNlLnB5IjogImQ2MTkyYmUzZTA1ZGFjZWI3NzU2YTM1MmE0ZjQzNmQzNTQ0MjJhOWIyOWU0YjA5OGZmNDI3MjJkMDM3OGVmZDQiLAogICAgInNvbHV0aW9uL2NvbXBhY3RfbWxwX2luc3RhbGwucHkiOiAiN2MyMGE4ZDI2ZDg5YmRiZjAyN2MwMzczNzE5YjIwYzc5MTE3ODY1MDZiZDY1NDkyOGUxN2QzYzE2YjA3MjQ5YSIsCiAgICAic29sdXRpb24vZm9yd2FyZF9vYnNlcnZlci5weSI6ICJjZDJjYTZiNmQ1MWM1NmJiNGIzYWMwNTVhOGU3NGIzYmU2ZDMxMWQzZmNmZGQ2ZTY5ZjgwNDliMGUyNDYwZTljIiwKICAgICJzb2x1dGlvbi9tbHBfaGFsZi9fX2luaXRfXy5weSI6ICJjOGUyNWM4M2RhNzc2NDljN2U3NmQ3ZTI1NTdkY2E4YTNiOWU5NzVhODFmYTczODcxMzIxMWE5YThmOGQ2ZmYwIiwKICAgICJzb2x1dGlvbi9tbHBfaGFsZi9uYXRpdmVfb3duZXIucHkiOiAiN2I2OWYyYTM1NGRlZjI1Y2I1M2Q0Y2E3OGYzYzE5MDgyNDk3ODM4YzVmNmVlMTMzZDk0Njk0ZTYxMTNkNTQwYyIsCiAgICAic29sdXRpb24vbWxwX2hhbGYvb3duZWRfaGFsZi5weSI6ICI4YTcxNDIwNWM0ZjFkZThmZjc5OTdmOTQzMGU0N2ZiNDE3YjU0ZTY2ODc5YjY0ZmE4MDE0NDg5NmM1MmE4MDdiIiwKICAgICJzb2x1dGlvbi9tbHBfaGFsZi9zdGFydGVyX2hhbGYucHkiOiAiMzc1OWYzNzYxYjNiM2QyYThlMDE4YjgwYTg1NzFjOWNjMTQ1NmI4YWRjMWVkNDBlNDdlMWM5MTkxMzJlN2RkYSIsCiAgICAic29sdXRpb24vbWxwX2hhbGYvc3RhcnRlcl9zd2lnbHUucHkiOiAiZTY2NTkxMGZhZWNhNjA5MWM5NThlZDdhMTkxNjA0NDg0YTQ0MjE2ZjUxZDZmYTA2OWY1ZjQ4MjIwZTc0NzJlNiIsCiAgICAic29sdXRpb24vbWxwX29ic2VydmF0aW9uX3Nlc3Npb24ucHkiOiAiNGZjZWJjY2JiMjkxZGJlNjBjMTcwZjA1MDNkNGYyNmFmYzVmYTc3ZjQ5ZDBiODNjNWU5ODJiMTgwNjczODNlNCIsCiAgICAic29sdXRpb24vcm93X2Fkam9pbnQucHkiOiAiNTA4MTRjNjQ5M2UxZDI0MWMwNmMyZDljYzc4NmMxNzZjZDA0MTUzMzYxYjVjZGIyZDY0NmNhYWQ0NjFjMDM4NCIsCiAgICAic29sdXRpb24vdHJhaW5lcl9zaGFkb3cucHkiOiAiYWRkYmViYTc2MmUzYWEzNjZmMjUzYmRkZDljZDlhOWM4Njg0ZjFjNGIzNTRhZGZiMzk2NjRiZmE4MzlkNGE0NCIsCiAgICAid29ya2VyLnB5IjogIjA2MDU3NmFmOGRiYWI4OTNkODRlOGMwMWZjY2NlMTM5ZjA2M2FjZWMzYTljZWY0MTk2YTE2OGI2ZmQ1ZGFmMjciCiAgfQp9Cg=='}
PAYLOAD_SHA256 = {'runtime_probe.py': 'efe1b7cf3f3cf1e37bbc169a8468859c5127ec3281a37e39068703b2eb3c1480', 'process_guard.py': '82465943a3a9af3831a5399a3a55d09ac290ba7365c4e80d6bc5e79fab62a6ff', 'owned_process_tree.py': '693dde708da5c06053af239463952115ef5e8865f50bcbc51a2038237ca57d69', 'prepare_asset_manifest.py': 'f2a0f36b5cd367d506c831c69f1a39d49c38c3c815758be5d0fd7a2ec033f46d', 'official_manifest.json': 'd013e1633940c27d708e2474a8280990000e33018e889ed14655c856329ff1d8', 'runtime_api_reference.json': 'd79e555702154a0516efa87ca22ab00dad39c515eefb12fc7332a0da40108881', 'web_runner.py': '13d6cff08bca92fac48e4dd76813224f732e15c4eabe271dc566e2135c029b5d', 'diagnostic_sources.zip': '9d8d043c14732b0d96e335b4c6a1659d2f4cf6f01fc3d275bf49b8513743b5b6', 'worker_manifest.json': '79e117968aa116b741686f442b72b279ac84a7f0ec14bb9a211006077810d524'}
STAGED = Path(tempfile.mkdtemp(prefix='soup-web-'+MODE+'-', dir='/kaggle/working'))
for name, encoded in PAYLOAD.items():
    raw = base64.b64decode(encoded, validate=True)
    assert hashlib.sha256(raw).hexdigest() == PAYLOAD_SHA256[name], name
    (STAGED/name).write_bytes(raw)
print('Prepared mode:', MODE)
print('Official source will be verified by bytes, not .git.')
print('Requested diagnostic worker cap: 80 seconds; zero training-loop runs.')


## Execute one bounded native-return shadow diagnostic, only after resume
The exact native compute_loss runs once; all forward and metric operations finish before the shadow backward owner begins. Sources, original batch identity, full padded native outputs, process cleanup/quarantine and runtime guards remain blocking. One80-second worker,24persistent export-file cap, no raw-array fanout or automatic retry.


In [ ]:
import subprocess, sys, signal
# Executing/submitting this notebook is the GPU action. Preparation did not run it.
# Never force-kill this supervisor: its guardian owns descendant cleanup and lock.
process = subprocess.Popen([sys.executable, '-B', str(STAGED/'web_runner.py'), '--mode', MODE], start_new_session=True)
try:
    return_code = process.wait()
except BaseException:
    if process.poll() is None:
        process.send_signal(signal.SIGTERM)
    while process.poll() is None:
        try: process.wait()
        except KeyboardInterrupt: pass
    raise
if return_code:
    raise RuntimeError('Screen blocked, failed, or censored. Inspect exported status/logs; do not treat it as a score or automatically retry.')
